# Final report

This notebook combines the former `1_data_overview_and_preprocessing`, `2_data_analysis` and `3_similarity` notebooks, in that order:

1. **Data overview and preprocessing:** inspects `data/catalog.json` and writes `data/features_en.json` and `data/products_en.json`.
2. **Data analysis:** statistics on the preprocessed feature names and products.
3. **Similarity matrix:** builds the shared similarity matrix and saves it to `data/similarity.npz` for all clustering notebooks.

Run it from the repository root. Part 3 downloads the Qwen embedding model (~1.2 GB) on first run.

# 1. Data overview and preprocessing

This notebook explores the raw `data/catalog.json` received from the company. We inspect its structure, product mix, feature-name frequency, and a reproducible sample of products and their feature descriptions. The examples use the catalog's default locale.

## 1.1 Load the catalog and inspect its structure

The catalog is nested: product fields live under `data`, localized text lives under `data.content[locale]`, and product features are stored as two parallel lists: `feature_titles` (the feature names) and `features` (the descriptions). The path check supports running this notebook either from this project folder or its parent workspace.

In [1]:
import html
import json
import random
from collections import Counter
from pathlib import Path
from pprint import pprint

catalog_path = Path("data/catalog.json")
if not catalog_path.is_file():
    catalog_path = Path("CTFDS-Project/data/catalog.json")
if not catalog_path.is_file():
    raise FileNotFoundError("Could not find data/catalog.json from the notebook or workspace folder.")

catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
products = catalog.get("products")
if not isinstance(products, list):
    raise ValueError("Expected catalog.json to contain a 'products' list.")

default_locale = catalog.get("default_locale", "en-GB")
print(f"Catalog: {catalog_path.resolve()}")
print(f"Schema version: {catalog.get('schema_version')}")
print(f"Products: {len(products)}")
print(f"Default locale: {default_locale}")
print(f"Locales: {catalog.get('locales', [])}")
print("Top-level keys:", list(catalog.keys()))
print("Keys in a product record:", list(products[0].keys()) if products else [])

Catalog: /sessions/rcw-01u1bd41xueqhlvmvih7pz9u/work/tmp/data/catalog.json
Schema version: 1.0.0
Products: 208
Default locale: en-GB
Locales: ['en-GB', 'de-DE', 'es-ES', 'fr-FR', 'it-IT', 'nl-NL', 'sv-SE', 'da-DK', 'nb-NO']
Top-level keys: ['schema_version', 'default_locale', 'locales', 'currencies', 'inheritance', 'products']
Keys in a product record: ['id', 'data', 'media', 'variants']


## 1.2 Product and feature statistics

Feature-name counts below describe each parent product's default-locale content and are exact after trimming whitespace and decoding HTML entities; capitalization and wording variants remain separate. We also summarize variants and check whether each product has the same number of feature names and descriptions, since the raw file appears to associate these by list position.

In [2]:
product_types = Counter()
genders = Counter()
locale_coverage = Counter()
feature_name_counts = Counter()
variant_total = 0
products_with_variants = 0
variants_with_default_locale_content_override = 0
product_summaries = []
length_mismatches = []
blank_feature_descriptions = []

for product in products:
    variants = product.get("variants") or []
    variant_total += len(variants)
    products_with_variants += bool(variants)
    for variant in variants:
        override_content = ((variant.get("overrides") or {}).get("content") or {})
        variants_with_default_locale_content_override += default_locale in override_content

    product_data = product.get("data") or {}
    classification = product_data.get("classification") or {}
    content_by_locale = product_data.get("content") or {}
    locale_coverage.update(content_by_locale.keys())
    content = content_by_locale.get(default_locale) or {}
    feature_names = content.get("feature_titles") or []
    descriptions = content.get("features") or []

    product_type = classification.get("product_type", "[missing]")
    gender = classification.get("gender", "[missing]")
    product_types[product_type] += 1
    genders[gender] += 1

    clean_feature_names = [html.unescape(str(feature_name)).strip() for feature_name in feature_names]
    clean_feature_names = [feature_name for feature_name in clean_feature_names if feature_name]
    feature_name_counts.update(clean_feature_names)
    product_summaries.append({
        "id": product.get("id", "[missing]"),
        "name": content.get("name", "[missing name]"),
        "product_type": product_type,
        "gender": gender,
        "feature_count": len(feature_names),
        "description_count": len(descriptions),
    })

    if len(feature_names) != len(descriptions):
        length_mismatches.append({
            "id": product.get("id", "[missing]"),
            "name": content.get("name", "[missing name]"),
            "feature_names": len(feature_names),
            "descriptions": len(descriptions),
        })
    for index, feature_name in enumerate(feature_names):
        if index >= len(descriptions) or not str(descriptions[index]).strip():
            blank_feature_descriptions.append((product.get("id"), feature_name))

print(f"Parent products: {len(products)}")
print(f"Variants: {variant_total} across {products_with_variants} products")
print(f"Variants with a {default_locale} content override: {variants_with_default_locale_content_override}")
print(f"Localized product-content records: {dict(sorted(locale_coverage.items()))}")
print(f"Feature-name occurrences: {sum(feature_name_counts.values())}")
print(f"Distinct exact feature names: {len(feature_name_counts)}")
print(f"Products with different feature-name/description counts: {len(length_mismatches)}")
print(f"Feature names without a non-empty description at the same position: {len(blank_feature_descriptions)}")

print("\nProducts by type:")
for name, count in product_types.most_common():
    print(f"  {name}: {count}")

print("\nProducts by gender:")
for name, count in genders.most_common():
    print(f"  {name}: {count}")

print("\nMost frequent feature names:")
for feature_name, count in feature_name_counts.most_common(15):
    print(f"  {count:>3}  {feature_name}")

Parent products: 208
Variants: 444 across 208 products
Variants with a en-GB content override: 72
Localized product-content records: {'da-DK': 208, 'de-DE': 208, 'en-GB': 208, 'es-ES': 208, 'fr-FR': 208, 'it-IT': 208, 'nb-NO': 208, 'nl-NL': 208, 'sv-SE': 208}
Feature-name occurrences: 1073
Distinct exact feature names: 444
Products with different feature-name/description counts: 7
Feature names without a non-empty description at the same position: 13

Products by type:
  Cycling Wear Tops: 42
  Cycling Wear Bottoms: 31
  Socks: 25
  Headwear: 24
  Shoe Covers: 21
  Base Layers: 21
  Gloves Cycling Winter: 15
  Warmers: 12
  Gloves Cycling Summer Short: 7
  Gloves Cycling Summer Long: 6
  Accessories: 2
  Cycling Wear One-Pieces: 2

Products by gender:
  Unisex: 115
  Men's: 54
  Women's: 38
  Junior: 1

Most frequent feature names:
   53  Reflective details
   23  Highly breathable
   15  Durable water repellent (DWR) treatment
   15  Stretch arch support
   14  Anti-odour treatment
  

## 1.3 Inspect sample products and features

The sample is reproducible and includes a product with mismatched feature-list lengths when one exists. If the two lists have different lengths, the code prints them separately rather than using `zip`, which would silently drop unmatched entries and imply pairings we cannot verify.

In [3]:
sample_products = random.Random(42).sample(products, k=min(3, len(products)))
mismatch_ids = {row["id"] for row in length_mismatches}
mismatch_example = next((product for product in products if product.get("id") in mismatch_ids), None)
if mismatch_example is not None and all(p.get("id") != mismatch_example.get("id") for p in sample_products):
    sample_products.append(mismatch_example)

for product in sample_products:
    product_data = product.get("data") or {}
    classification = product_data.get("classification") or {}
    content = (product_data.get("content") or {}).get(default_locale) or {}
    feature_names = content.get("feature_titles") or []
    descriptions = content.get("features") or []

    print(f"\nProduct {product.get('id')} — {content.get('name', '[missing name]')}")
    print(f"Type: {classification.get('product_type', '[missing]')} | Gender: {classification.get('gender', '[missing]')}")
    variants = product.get("variants") or []
    print(f"Variants: {len(variants)}")
    for variant in variants[:2]:
        color = (variant.get("color") or {}).get("name", "[color not specified]")
        print(f"  Variant {variant.get('id', '[missing id]')} | color: {color}")
    if len(feature_names) == len(descriptions):
        for feature_name, description in zip(feature_names, descriptions):
            print(f"  - {feature_name}: {description or '[empty description]'}")
    else:
        print(f"  Feature names ({len(feature_names)}):")
        for feature_name in feature_names:
            print(f"    - {feature_name}")
        print(f"  Descriptions ({len(descriptions)}; pairing not assumed):")
        for description in descriptions:
            print(f"    - {description or '[empty description]'}")


Product 6111 — GripGrab RIDE Cycling Shorts
Type: Cycling Wear Bottoms | Gender: Men's
Variants: 1
  Variant 611101 | color: Black
  - High-quality construction: Crafted from high-quality fabrics offering a balanced level of stretch and support for enhanced comfort
  - Quality padding (LIEGE HP): Integrates a high-quality cycling pad from Elastic Interface (LIEGE HP) to deliver outstanding comfort during long rides
  - 1 top entry leg pocket: Features a convenient stretchy pocket, lying flat against the leg for easy access to your phone or snacks throughout your ride
  - Wide elastic hems: Ensure a secure fit on the legs with wide elastic hems and silicone grippers
  - Wicking waistband: Stays comfortably in place during your rides thanks to the stretch waistband lined with moisture-wicking mesh

Product 1075 — GripGrab Primavera 2 Merino Spring-Autumn Gloves
Type: Gloves Cycling Winter | Gender: Unisex
Variants: 1
  Variant 107501 | color: Black
  - Luxurious soft merino blend: A mer

In [4]:
print("Examples for the three most frequent feature names:")
for feature_name, _ in feature_name_counts.most_common(3):
    print(f"\n{feature_name}")
    shown = 0
    for product in products:
        product_data = product.get("data") or {}
        content = (product_data.get("content") or {}).get(default_locale) or {}
        feature_names = content.get("feature_titles") or []
        descriptions = content.get("features") or []
        if len(feature_names) != len(descriptions):
            continue
        for raw_feature_name, description in zip(feature_names, descriptions):
            if html.unescape(str(raw_feature_name)).strip() == feature_name:
                print(f"  {content.get('name', '[missing name]')}: {description or '[empty description]'}")
                shown += 1
                if shown == 3:
                    break
        if shown == 3:
            break

Examples for the three most frequent feature names:

Reflective details
  GripGrab Waterproof Helmet Cover: for visibility in the traffic
  GripGrab W's PACR Thermal Bib Shorts: For visibility in traffic
  GripGrab PACR Windproof Lightweight Vest: Enhance your visibility to other road users in low light conditions

Highly breathable
  GripGrab FastStream Aero Cycling Socks: The foot section of the sock, crafted from Coolmax® yarn, offers great breathability
  GripGrab RIDE Winter Long Sleeve Base Layer: providing excellent ventilation
  GripGrab Ultralight Mesh Sleeveless Base Layer 3-Pack: Ultralight open mesh fabric allows heat and moisture to escape from within, keeping you cool in hot summer conditions

Durable water repellent (DWR) treatment
  GripGrab W's PACR Thermal Bib Shorts: Providing effective water repellency
  GripGrab RIDE 2 Waterproof Winter Gloves: providing effective water repellency
  GripGrab RIDE 2 Windproof Spring-Autumn Gloves: providing effective water repellenc

## 1.4 Why preprocessing is needed and what it will do

The raw catalog is useful for product-level inspection, but it is not yet a clean feature table. Content is nested by product and locale; each product may also have variant-level overrides. Feature names and descriptions are stored in separate positional lists, and some products have different list lengths or empty descriptions. Repeated feature names also occur across products with different wording, capitalization, or HTML entities.

We will turn the catalog into two smaller files for the analysis that follows:

- `data/features_en.json`: one record per cleaned feature name, with its associated descriptions; descriptions that cannot safely be paired are kept separately.
- `data/products_en.json`: one record per product, with its product type and feature names, for later evaluation.

The JSON files keep the catalog's field names (`attribute_records`, `title`, `titles`); in the text we call these feature names.

The code selects the default English locale, includes product-level and variant-override content, decodes HTML entities, trims text, and aggregates repeated feature names. When a product's feature-name and description lists have different lengths, it does not guess the pairings: it keeps those descriptions as unpaired instead of silently losing them with `zip`.

## 1.5 Preprocess the catalog

Run this after the overview cells. The outputs are written next to `catalog.json` and are used by the following analysis and clustering notebooks.

In [5]:
import html
import json

data_dir = catalog_path.parent
features_path = data_dir / "features_en.json"
products_path = data_dir / "products_en.json"

blocks = []
for raw_product in products:
    product_type = raw_product["data"]["classification"]["product_type"]
    product_id = raw_product["id"]
    content = (raw_product.get("data") or {}).get("content") or {}
    blocks.append((product_id, product_type, content.get(default_locale, {})))

    for variant in raw_product.get("variants") or []:
        variant_content = ((variant.get("overrides") or {}).get("content") or {})
        blocks.append((product_id, product_type, variant_content.get(default_locale, {})))

feature_name_records = {}
product_records = {}
unpaired_descriptions = []

for product_id, product_type, content in blocks:
    feature_names = [html.unescape(feature_name).strip() for feature_name in content.get("feature_titles") or []]
    descriptions = [html.unescape(description).strip() for description in content.get("features") or []]

    product = product_records.setdefault(
        product_id,
        {"product_id": product_id, "product_type": product_type, "titles": set()},
    )
    product["titles"].update(feature_names)

    for feature_name in feature_names:
        feature_name_records.setdefault(feature_name, {"title": feature_name, "descriptions": []})

    if len(feature_names) != len(descriptions):
        unpaired_descriptions.extend(descriptions)
        continue

    for feature_name, description in zip(feature_names, descriptions):
        feature_name_records[feature_name]["descriptions"].append(description)

for product in product_records.values():
    product["titles"] = sorted(product["titles"])

features_path.write_text(
    json.dumps(
        {
            "attribute_records": list(feature_name_records.values()),
            "unpaired_descriptions": unpaired_descriptions,
        },
        ensure_ascii=False,
        indent=1,
    ),
    encoding="utf-8",
)
products_path.write_text(
    json.dumps(list(product_records.values()), ensure_ascii=False, indent=1),
    encoding="utf-8",
)

print(f"Wrote {len(feature_name_records)} unique feature names to {features_path}")
print(f"Kept {len(unpaired_descriptions)} descriptions that could not be paired safely")
print(f"Wrote {len(product_records)} products to {products_path}")

Wrote 445 unique feature names to data/features_en.json
Kept 22 descriptions that could not be paired safely
Wrote 208 products to data/products_en.json


# 2. Data analysis

Do we get any insight after preprocessing?

We examine feature names per product, feature-name frequency, descriptions, and how often pairs of feature names co-occur on the same product.

In [6]:
import json
import statistics
from collections import Counter
from itertools import combinations
from pathlib import Path

records = json.loads(Path("data/features_en.json").read_text(encoding="utf8"))["attribute_records"]
products = json.loads(Path("data/products_en.json").read_text(encoding="utf8"))
print(len(records), "feature names,", len(products), "products")

# Feature names per product
per_product = [len(p["titles"]) for p in products]
print(f"\nFeature names per product: mean {statistics.mean(per_product):.1f}, median {statistics.median(per_product)}, "
      f"min {min(per_product)}, max {max(per_product)}")

# Feature-name frequency: on how many products each feature name appears
feature_name_counts = Counter(t for p in products for t in p["titles"])
print(f"\n{sum(1 for c in feature_name_counts.values() if c == 1)} of {len(feature_name_counts)} feature names appear on only one product")
print(f"{sum(1 for c in feature_name_counts.values() if c >= 5)} feature names appear on 5 or more products")

# Most common feature names
print("\nMost common feature names:")
for feature_name, count in feature_name_counts.most_common(10):
    print(f"{count:3}  {feature_name}")

# Descriptions
desc_counts = [len(r["descriptions"]) for r in records]
distinct_counts = [len(set(r["descriptions"])) for r in records]
print(f"\n{sum(1 for c in desc_counts if c == 0)} feature names have no description")
print(f"mean {statistics.mean(desc_counts):.1f} descriptions per feature name, but only {statistics.mean(distinct_counts):.1f} distinct")

# Pairs of feature names that appear on the same product 
# TODO (TO BE IMPROVED with APRIORI LOGIC)
same_product_pairs = set()
for p in products:
    same_product_pairs.update(combinations(p["titles"], 2))
n = len(records)
all_pairs = n * (n - 1) // 2
print(f"\nSame-product feature-name pairs: {len(same_product_pairs)} of {all_pairs} possible ({len(same_product_pairs) / all_pairs:.1%})")

445 feature names, 208 products

Feature names per product: mean 5.2, median 5.0, min 3, max 11

202 of 445 feature names appear on only one product
48 feature names appear on 5 or more products

Most common feature names:
 53  Reflective details
 23  Highly breathable
 15  Durable water repellent (DWR) treatment
 15  Stretch arch support
 14  Anti-odour treatment
 12  Reflective branding
 11  Race fit
 10  Moisture wicking
  9  COOLMAX® fibre
  9  Fast-drying material

12 feature names have no description
mean 2.3 descriptions per feature name, but only 1.4 distinct

Same-product feature-name pairs: 1442 of 98790 possible (1.5%)


## 2.1 Visualizing product and feature patterns

These plots explore the preprocessed catalog only: how many feature names products have, how often each feature name is reused across products, and how many description entries are collected for each feature name.

A feature name can have several description entries because it appears on several products. Some entries may contain identical text. The final plot counts entries, not unique wording; the examples below the chart show what those counts mean.

In [7]:
import plotly.graph_objects as go
from IPython.display import display

# How many distinct feature names are used by 1, 2, 3, 4, or 5+ products?
support_categories = ["1", "2", "3", "4", "5+"]
support_group = [str(count) if count < 5 else "5+" for count in feature_name_counts.values()]
support_category_counts = [sum(group == category for group in support_group) for category in support_categories]

support_figure = go.Figure(
    go.Bar(
        x=support_categories,
        y=support_category_counts,
        text=[f"{count} ({count / len(feature_name_counts):.1%})" for count in support_category_counts],
        textposition="outside",
        marker_color="#3d7899",
        hovertemplate="Appears on %{x} product(s)<br>Feature names: %{y}<extra></extra>",
    )
)
support_figure.update_layout(
    title="How often is each feature name used across products?",
    xaxis_title="Number of products containing a feature name",
    yaxis_title="Number of distinct feature names",
    template="simple_white",
    showlegend=False,
)
display(support_figure)

# Distribution of how many feature names each product has.
product_feature_name_counts = Counter(len(product["titles"]) for product in products)
product_feature_name_values = sorted(product_feature_name_counts)
product_figure = go.Figure(
    go.Bar(
        x=product_feature_name_values,
        y=[product_feature_name_counts[count] for count in product_feature_name_values],
        text=[product_feature_name_counts[count] for count in product_feature_name_values],
        textposition="outside",
        marker_color="#62a386",
        hovertemplate="Feature names per product: %{x}<br>Products: %{y}<extra></extra>",
    )
)
product_figure.update_layout(
    title="How many feature names does each product have?",
    xaxis_title="Number of feature names",
    yaxis_title="Number of products",
    template="simple_white",
    showlegend=False,
)
display(product_figure)

# Count description entries attached to each feature name.
description_categories = ["0", "1", "2", "3", "4", "5+"]
description_bins = Counter(str(count) if count < 5 else "5+" for count in desc_counts)

description_figure = go.Figure(
    go.Bar(
        x=description_categories,
        y=[description_bins[category] for category in description_categories],
        text=[description_bins[category] for category in description_categories],
        textposition="outside",
        marker_color="#d18b45",
        hovertemplate="Description entries for a feature name: %{x}<br>Feature names: %{y}<extra></extra>",
    )
)
description_figure.update_layout(
    title="How many description entries are attached to each feature name?",
    xaxis_title="Description entries per feature name (5+ grouped)",
    yaxis_title="Number of feature names",
    template="simple_white",
    showlegend=False,
)
display(description_figure)

# Concrete examples: show how repeated entries are stored for individual feature names.
example_feature_names = ["Light insulation", "Reflective details"]
for feature_name in example_feature_names:
    record = next((record for record in records if record["title"] == feature_name), None)
    if record is None:
        continue
    descriptions = Counter(record["descriptions"])
    print(
        f"\n{feature_name}: {sum(descriptions.values())} description entries, "
        f"{len(descriptions)} distinct text versions"
    )
    for description, count in descriptions.most_common(3):
        print(f"  {count}x: {description}")


Light insulation: 5 description entries, 3 distinct text versions
  3x: Delivering warmth and exceptional breathability for comfort during transitional rides
  1x: Soft knitted fabric offers warmth retention and protection from cooler spring-autumn temperatures and windchill
  1x: The Thermolite® fabric and padded sole provide light insulation for cool spring and autumn days

Reflective details: 53 description entries, 14 distinct text versions
  12x: for visibility in traffic
  9x: Enhance your visibility to other road users in low light conditions
  8x: For visibility in traffic


## 2.2 Example: feature names and descriptions for one product

In the raw catalog, each product has a list of feature names and a parallel list of descriptions. When the two lists have equal lengths, an item at the same position in each list is a feature-name/description pair. The example below prints one complete product record. Across the full catalog, the same feature name can occur on multiple products with different descriptions; preprocessing groups those description entries together by feature name.

In [8]:
import html
import json
from pathlib import Path

catalog_path = Path("data/catalog.json")
if not catalog_path.is_file():
    catalog_path = Path("CTFDS-Project/data/catalog.json")
if not catalog_path.is_file():
    raise FileNotFoundError("Could not find data/catalog.json.")

catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
locale = catalog.get("default_locale", "en-GB")
sample = None
for product in catalog.get("products", []):
    product_data = product.get("data") or {}
    content = (product_data.get("content") or {}).get(locale) or {}
    feature_names = content.get("feature_titles") or []
    descriptions = content.get("features") or []
    if feature_names and len(feature_names) == len(descriptions):
        sample = (product, product_data, content, feature_names, descriptions)
        break

if sample is None:
    raise ValueError("No product with aligned feature names and descriptions was found.")

product, product_data, content, feature_names, descriptions = sample
classification = product_data.get("classification") or {}
print(f"Product {product.get('id')}: {content.get('name', '[missing name]')}")
print(f"Product type: {classification.get('product_type', '[missing]')}\n")

for feature_name, description in zip(feature_names, descriptions):
    print(f"Feature name: {html.unescape(str(feature_name)).strip()}")
    print(f"Description: {html.unescape(str(description)).strip()}\n")

Product 4023: GripGrab PACR Spring-Autumn Leg Warmers
Product type: Warmers

Feature name: Light insulation
Description: Delivering warmth and exceptional breathability for comfort during transitional rides

Feature name: DWR coating
Description: Delivering effective water repellency in wet conditions

Feature name: Pre-shaped fit
Description: Delivering a precise, anatomical fit that follows the natural contours of the legs

Feature name: Secure hold with elasticated grippers
Description: Keeping the leg warmers securely in place

Feature name: Reflective branding
Description: Adding a touch of reflectivity



# 3. Similarity matrix

We built one similarity score for every pair of the 445 feature names, used by all clustering notebooks.

1. Load `data/features_en.json` from Part 1 above.
2. TF-IDF similarity of the descriptions.
3. Semantic similarity from Qwen embeddings of feature name + descriptions.
4. Blend with fuzzy feature-name matching (50% semantic, 25% TF-IDF, 25% fuzzy).
5. Penalise pairs with conflicting concepts and save the matrix to `data/similarity.npz`.

**Input:** `data/features_en.json` from Part 1 above, with one record per unique feature name and its descriptions. The unpaired descriptions are only used for the TF-IDF word weights.

**Dependencies:** see `requirements.txt` (Python 3.13). The first run downloads the Qwen model (~1.2 GB).

In [1]:
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
from rapidfuzz import fuzz, process, utils
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# weights of the three similarity signals
W_SEMANTIC, W_TFIDF, W_FUZZY = 0.50, 0.25, 0.25
W_NAME = 0.50
PENALTY = 0.30
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
EMBEDDING_PROMPT = "Instruct: Identify product features that describe the same property\nQuery: "

W1121 23:07:26.430000 1366 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 3.1 Load the feature records

Loads the 445 feature names and their descriptions from `data/features_en.json`. The 22 unpaired descriptions are kept separately and only used for the TF-IDF word weights.

In [2]:
feature_data = json.loads(Path("data/features_en.json").read_text(encoding="utf8"))
feature_name_records = feature_data["attribute_records"]
unpaired_descriptions = feature_data["unpaired_descriptions"]

feature_names = [record["title"] for record in feature_name_records]
descriptions = [record["descriptions"] for record in feature_name_records]

print(len(feature_names), "feature names,", sum(bool(d) for d in descriptions), "with descriptions,",
      len(unpaired_descriptions), "unpaired descriptions")

445 titles, 433 with descriptions, 22 unpaired descriptions


## 3.2 TF-IDF similarity of the descriptions

Each feature name's descriptions are joined into one document. TF-IDF gives rare words more weight than words used everywhere. The word weights are learned from all descriptions but only feature names get a row in the matrix. Feature names without descriptions get 0 here and rely on the other two signals.

In [3]:
#here, each feature name has its document
documents = [" ".join(texts) for texts in descriptions]

# word weights are learned from all descriptions
tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True)
tfidf.fit([d for d in documents if d] + unpaired_descriptions)

# rows are normalised, so the dot product is the cosine similarity
vectors = tfidf.transform(documents)
tfidf_similarity = (vectors @ vectors.T).toarray()
print("TF-IDF similarity matrix:", tfidf_similarity.shape)

TF-IDF similarity matrix: (445, 445)


## 3.3 Semantic feature-name-and-description embeddings

An embedding is a vector (a point in a high-dimensional meaning space) produced from a piece of text. Texts with related meanings tend to have vectors pointing in similar directions, even when they use different words. This step creates one semantic vector for each feature name; it does not cluster the feature names yet.

For every feature name, the model embeds the feature name and each distinct, aligned description. If a feature name has several descriptions, their vectors are averaged using the number of times each description occurs in the data. The feature-name vector and this description-context vector are then blended equally (`W_NAME = 0.50`) and normalized. Later, the cosine similarity between two resulting vectors gives their raw semantic similarity.

**Why cosine similarity?** The vectors are normalized to length 1, so their dot product equals cosine similarity: `cos(theta) = (u · v) / (||u|| ||v||) = u · v`. Cosine compares vector direction (semantic orientation) rather than length. “Raw” means this score is calculated before the median-based calibration later in this notebook.

**Other possible comparisons:** Euclidean distance (`||u - v||`) is a natural alternative. For unit-length vectors, it gives the same ranking as cosine because `||u - v||^2 = 2(1 - cosine_similarity)`; only the distance scale changes. Manhattan distance (`sum(abs(u - v))`) is another option, but it measures separation differently and may rank pairs differently, so it should be checked against reviewed similar/dissimilar pairs of feature names. A plain dot product is also possible, but without normalization it is sensitive to vector length; with these normalized vectors it is identical to cosine similarity.

**Example — “Light insulation”:** this feature name has five description entries but three distinct texts. “Delivering warmth and exceptional breathability for comfort during transitional rides” occurs three times; the other two descriptions occur once each. The description-context vector is therefore conceptually `(3 × vector(description 1) + vector(description 2) + vector(description 3)) / 5`. That context is blended 50/50 with the embedding of the feature name “Light insulation”. The repeated description pulls the context toward its meaning, but does not change the feature name’s own embedding.

If a feature name has no aligned description, its semantic vector is just its own embedding—not a zero vector. Descriptions that could not safely be matched to feature names are excluded from this step. They are used only when fitting TF-IDF in the next section. The public model may need to download the first time; no access token or login is required.

In [4]:
import torch

embedding_model = SentenceTransformer(MODEL_NAME, model_kwargs={"dtype": torch.float32})
name_vectors = embedding_model.encode(
    feature_names,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
    show_progress_bar=True,
)

all_descriptions = sorted({text for texts in descriptions for text in texts})
description_embedding_vectors = embedding_model.encode(
    all_descriptions,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
    show_progress_bar=True,
)
description_vector_by_text = {
    text: description_embedding_vectors[index]
    for index, text in enumerate(all_descriptions)
}

feature_name_vectors = []
for name_vector, texts in zip(name_vectors, descriptions):
    if texts:
        description_counts = Counter(texts)
        weighted_description_vector = np.average(
            [description_vector_by_text[text] for text in description_counts],
            axis=0,
            weights=list(description_counts.values()),
        )
        vector = W_NAME * name_vector + (1 - W_NAME) * weighted_description_vector
        vector = vector / np.linalg.norm(vector)
    else:
        vector = name_vector
    feature_name_vectors.append(vector)

feature_name_vectors = np.asarray(feature_name_vectors)
raw_semantic = feature_name_vectors @ feature_name_vectors.T
baseline_similarity = np.median(raw_semantic[np.triu_indices(len(feature_names), k=1)])
if baseline_similarity >= 1:
    raise ValueError("Semantic similarities cannot be calibrated: median baseline is 1.")

# Calibrate model-specific cosine baselines to 0; identical vectors remain at 1.
semantic = np.clip(
    (raw_semantic - baseline_similarity) / (1 - baseline_similarity),
    0,
    1,
)
print(f"Semantic matrix ready; uncalibrated median baseline: {baseline_similarity:.3f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Semantic matrix ready; uncalibrated median baseline: 0.566


## 3.4 Blend the three similarities

Fuzzy matching compares the wording of the feature names, ignoring word order ("Cycling Gloves" = "Gloves Cycling"). The final similarity is a weighted average: 50% semantic, 25% TF-IDF and 25% fuzzy. We blend the similarity matrices instead of combining the vectors, because embeddings and TF-IDF vectors live in different spaces.

In [5]:
assert np.isclose(W_SEMANTIC + W_TFIDF + W_FUZZY, 1), "weights must sum to 1"

fuzzy = process.cdist(
    feature_names,
    feature_names,
    scorer=fuzz.token_sort_ratio,
    processor=utils.default_process,
) / 100

#rescaling fuzzy the same way as semantic
fuzzy_baseline = np.median(fuzzy[np.triu_indices(len(feature_names), k=1)])
fuzzy = np.clip((fuzzy - fuzzy_baseline) / (1 - fuzzy_baseline), 0, 1)

similarity = (
    W_SEMANTIC * semantic
    + W_TFIDF *  tfidf_similarity
    + W_FUZZY * fuzzy
)
# A missing description gives no IDF evidence, but every item has zero distance to itself.
np.fill_diagonal(similarity, 1.0)
print(f"Blended similarity matrix: {similarity.shape}")

Blended similarity matrix: (445, 445)


## 3.5 Penalty for conflicting concepts

Embeddings see "Deep winter protection" and "Windproof protection" as similar, but they describe different properties (insulation vs. blocking wind). Pairs where one feature name is only about winter/cold and the other only about wind lose 0.30 similarity (72 pairs). Feature names that mention both are not penalised.


> **TODO:** this rule was carried over from an earlier notebook so we need to check it. We need to compare clustering results with and without the penalty once the evaluation exists

In [6]:
CONCEPTS = {
    "winter": r"\b(winter|cold|freez\w*|frost\w*)\b",
    "windproof": r"\b(wind\w*)\b",
}
CONFLICTS = [("winter", "windproof")]

concepts_by_feature_name = [
    {
        name
        for name, pattern in CONCEPTS.items()
        if re.search(pattern, feature_name, re.IGNORECASE)
    }
    for feature_name in feature_names
]

penalty_matrix = np.zeros_like(similarity)
for first_concept, second_concept in CONFLICTS:
    first_only = np.array(
        [first_concept in concepts and second_concept not in concepts for concepts in concepts_by_feature_name]
    )
    second_only = np.array(
        [second_concept in concepts and first_concept not in concepts for concepts in concepts_by_feature_name]
    )
    penalty_matrix[np.ix_(first_only, second_only)] = PENALTY
    penalty_matrix[np.ix_(second_only, first_only)] = PENALTY

similarity = np.clip(similarity - penalty_matrix, 0, 1)
print(f"Conflicting feature-name pairs penalized: {int(np.count_nonzero(penalty_matrix) // 2)}")

Conflicting attribute pairs penalized: 72


In [7]:

# Save the shared similarity matrix for all clustering notebooks
output_path = Path("data/similarity.npz")
output_path.parent.mkdir(exist_ok=True)

np.savez_compressed(
    output_path,
    similarity=similarity,                 # 445 x 445, values in [0, 1]
    titles=np.array(feature_names),           # row/column order of the matrix
)
print(f"Saved {similarity.shape} similarity matrix to {output_path}")

Saved (445, 445) similarity matrix to data/similarity.npz


# 4. General clustering

## 4.1 Agglomerative clustering

## 1. Agglomerative clustering and cluster label assignment

Agglomerative clustering starts with one cluster per feature name and merges clusters according to average pairwise distance. It accepts distances, not similarities, so the blended values are converted with `distance = 1 - similarity`. `n_clusters=None` means the number of groups is not fixed in advance; `distance_threshold = 0.70` controls when merging stops. A lower threshold generally produces more, smaller groups. `labels[i]` is the cluster label of `feature_names[i]`; the number itself has no semantic ranking.

In [1]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from sklearn.cluster import AgglomerativeClustering
from sklearn.manifold import TSNE

THRESHOLD = 0.70
HIGHLIGHT_GROUPS = 8

with np.load("data/similarity.npz") as data:
    similarity = data["similarity"]
    feature_names = data["titles"].tolist()

distance_matrix = 1 - similarity
model = AgglomerativeClustering(
    n_clusters=None,
    metric="precomputed",
    linkage="average",
    distance_threshold=THRESHOLD,
)
labels = model.fit(distance_matrix).labels_

sizes = Counter(labels)
multi = [label for label, size in sizes.items() if size > 1]
print(f"{len(sizes)} clusters: {len(multi)} with multiple feature names, {len(sizes) - len(multi)} singletons")

# Print members so each cluster can be inspected before the visualization.
members_by_cluster = {
    label: [feature_names[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in sizes
}
for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
    if len(members) > 1:
        print(f"\nCluster {label} ({len(members)} feature names)")
        for member in members:
            print(f"  - {member}")

138 clusters: 85 with multiple attributes, 53 singletons

Cluster 14 (20 attributes)
  - Highly breathable
  - Breathable mesh back
  - Insulating & breathable
  - Breathable
  - Breathable, windproof fabric
  - Cool and comfortable
  - Breathable construction
  - Extremely breathable
  - Super breathable
  - Ideal for hot conditions
  - Thermal insulation with exceptional breathability
  - Insulating & Breathable
  - Breathable mesh inserts
  - Breathable insulated backhand
  - Extremely breathable 40K (40,000 g/m²/24h)
  - Maximum breathability
  - Breathable 15K
  - Breathable mesh back panel
  - Breathable fabric
  - Ultralight and breathable

Cluster 0 (15 attributes)
  - Secure hold with elasticated grippers
  - Secure silicone gripper hem
  - Elastic silicone grippers
  - Silicone palm grip
  - Silicone gripper
  - InsideGrip™ technology
  - Silicone-enhanced palm sections
  - Performance grip
  - Secure grip and control
  - Silicone fingertips
  - Silicone Hem Gripper
  - Silic

## Clustering evaluation

_Reserved for evaluation later. No evaluation metrics or methods are calculated in this notebook._

## 2. Cluster visualization

t-SNE projects the same precomputed distances to two dimensions for display only; it does not change the cluster labels or serve as an evaluation metric. Nearby points suggest similarity in this projection, but the plot can distort some distances. The largest groups with several feature names receive distinct colors, other groups are muted, and singleton clusters are shown as hollow markers. Hover over a point to see its feature name and group. The fixed perplexity of 30 requires more than 30 feature names.

In [2]:
import importlib
import plotly.io._renderers as plotly_renderers

importlib.reload(plotly_renderers)

<module 'plotly.io._renderers' from '/Users/name/Desktop/CTFDS Project/.venv/lib/python3.14/site-packages/plotly/io/_renderers.py'>

In [3]:
from IPython.display import HTML, display

HIGHLIGHT_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY, INK, MUTED = "#8c8c86", "#c9c8c0", "#1f2933", "#6b7785"

coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    perplexity=30,
    random_state=42,
).fit_transform(distance_matrix)


def group_name(label):
    """Use the most centrally similar feature name as the group's display name."""
    indices = np.where(labels == label)[0]
    within_group_similarity = similarity[np.ix_(indices, indices)]
    medoid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return feature_names[medoid_index]


ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]
group_names = {label: group_name(label) for label in multi}


def make_trace(indices, name, color, hollow=False, size=9):
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[feature_names[index] for index in indices],
        customdata=[group_names.get(labels[index], "Singleton") for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )


other_group_indices = np.array(
    [index for index, label in enumerate(labels) if sizes[label] > 1 and label not in highlighted_groups]
)
singleton_indices = np.array([index for index, label in enumerate(labels) if sizes[label] == 1])

figure = go.Figure()
figure.add_trace(
    make_trace(singleton_indices, f"Singletons ({len(singleton_indices)})", LIGHT_GRAY, hollow=True, size=7)
)
figure.add_trace(
    make_trace(other_group_indices, f"Other groups ({len(ranked_groups) - len(highlighted_groups)})", GRAY, size=7)
)
for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    figure.add_trace(make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, size=10))

figure.update_layout(
    title=dict(text="Feature-name clusters: context-aware similarity (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

## 4.2 DBSCAN

# Generalized DBSCAN clustering

Clusters all 445 feature titles together, without filtering by product type.

Like the other clustering notebooks, this one loads the shared similarity matrix from `data/similarity.npz` (saved by `3_similarity.ipynb`). This keeps the input fixed, so result differences come from the algorithm itself.

**Method**:
1. Convert similarity to distance (`distance = 1 - similarity`).
2. Try a compact grid of DBSCAN parameters (`eps`, `min_samples`).
3. Select the configuration with the best silhouette score, counting noise points as singleton clusters.
4. Inspect clusters and visualize with t-SNE.
5. Check parameter sensitivity (k-distance plot, grid heatmaps).

In [1]:
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

EPS_VALUES = np.round(np.arange(0.20, 0.82, 0.02), 2)
MIN_SAMPLES_VALUES = [2, 3, 4, 5, 6, 8]
SEED = 0
HIGHLIGHT_GROUPS = 8

with np.load("data/similarity.npz") as data:
    similarity = data["similarity"]
    attributes = data["titles"].tolist()

n = len(attributes)
assert similarity.shape == (n, n)
assert np.allclose(similarity, similarity.T), "similarity must be symmetric"

distance_matrix = 1 - similarity
distance_matrix = np.clip(distance_matrix, 0, 1)
np.fill_diagonal(distance_matrix, 0)

print(f"Loaded {n} attributes, matrix {similarity.shape}")

Loaded 445 attributes, matrix (445, 445)


## 1. Parameter search

For each `(eps, min_samples)` pair, we fit DBSCAN on the precomputed distance matrix.

Parameters are selected by the silhouette score computed on all attributes, with each noise point treated as its own singleton cluster (silhouette 0). This prevents a configuration from scoring well by discarding difficult attributes as noise. Ties are broken by higher coverage, then fewer clusters. Both methods search the same `min_samples` range; DBSCAN additionally searches `eps` and HDBSCAN `min_cluster_size`. HDBSCAN has no `eps` parameter.

The silhouette on non-noise points only is reported for reference but is not used for selection.

In [2]:
def noise_as_singletons(labels):
    relabeled = labels.copy()
    noise_indices = np.where(relabeled == -1)[0]
    start = relabeled.max() + 1
    relabeled[noise_indices] = np.arange(start, start + len(noise_indices))
    return relabeled


def evaluate(labels, D):
    is_noise = labels == -1
    n_noise = int(is_noise.sum())
    n_clusters = len(set(labels[~is_noise].tolist()))
    if n_clusters < 2:
        return None

    singleton_labels = noise_as_singletons(labels)
    if len(set(singleton_labels.tolist())) >= len(labels):
        return None
    score = silhouette_score(D, singleton_labels, metric="precomputed")

    non_noise_labels = labels[~is_noise]
    if n_clusters < len(non_noise_labels):
        silhouette_non_noise = silhouette_score(
            D[np.ix_(~is_noise, ~is_noise)], non_noise_labels, metric="precomputed"
        )
    else:
        silhouette_non_noise = float("nan")

    return {
        "score": float(score),
        "silhouette_non_noise": float(silhouette_non_noise),
        "coverage": float((~is_noise).mean()),
        "n_clusters": int(n_clusters),
        "n_noise": n_noise,
    }


def format_row(row, param_names):
    params = ", ".join(f"{name}={row[name]:g}" for name in param_names)
    return (
        f"{params} | "
        f"score={row['score']:.4f}, silhouette_non_noise={row['silhouette_non_noise']:.4f}, "
        f"clusters={row['n_clusters']}, coverage={100 * row['coverage']:.1f}%, noise={row['n_noise']}"
    )


def warn_if_at_grid_edge(best, grids):
    for name, grid in grids.items():
        if np.isclose(best[name], min(grid)) or np.isclose(best[name], max(grid)):
            print(f"Warning ({name}={best[name]:g}): Selected value is at the edge of the grid - consider extending the grid.")


PARAM_GRIDS = {"eps": EPS_VALUES, "min_samples": MIN_SAMPLES_VALUES}

candidates = []
for eps in EPS_VALUES:
    for min_samples in MIN_SAMPLES_VALUES:
        model = DBSCAN(eps=float(eps), min_samples=min_samples, metric="precomputed")
        labels = model.fit_predict(distance_matrix)
        evaluation = evaluate(labels, distance_matrix)
        if evaluation is None:
            continue
        candidates.append({"eps": float(eps), "min_samples": int(min_samples), **evaluation})

if not candidates:
    raise ValueError("No valid DBSCAN configuration produced at least 2 non-noise clusters.")

candidates = sorted(candidates, key=lambda r: (-r["score"], -r["coverage"], r["n_clusters"]))
best = candidates[0]

print("Top 10 parameter sets (score = silhouette with noise as singletons):")
for row in candidates[:10]:
    print(format_row(row, PARAM_GRIDS))

print("\nSelected configuration:")
print(format_row(best, PARAM_GRIDS))
warn_if_at_grid_edge(best, PARAM_GRIDS)

Top 10 parameter sets (score = silhouette with noise as singletons):
eps=0.4, min_samples=2 | score=0.1380, silhouette_non_noise=0.4755, clusters=59, coverage=37.1%, noise=280
eps=0.38, min_samples=2 | score=0.1323, silhouette_non_noise=0.4946, clusters=56, coverage=34.4%, noise=292
eps=0.42, min_samples=2 | score=0.1265, silhouette_non_noise=0.4172, clusters=61, coverage=41.8%, noise=259
eps=0.36, min_samples=2 | score=0.1225, silhouette_non_noise=0.5323, clusters=50, coverage=29.7%, noise=313
eps=0.44, min_samples=2 | score=0.1220, silhouette_non_noise=0.3957, clusters=59, coverage=44.7%, noise=246
eps=0.46, min_samples=2 | score=0.1180, silhouette_non_noise=0.3604, clusters=58, coverage=48.3%, noise=230
eps=0.5, min_samples=2 | score=0.1163, silhouette_non_noise=0.3027, clusters=60, coverage=56.2%, noise=195
eps=0.34, min_samples=2 | score=0.1108, silhouette_non_noise=0.5663, clusters=43, coverage=24.9%, noise=334
eps=0.48, min_samples=2 | score=0.1094, silhouette_non_noise=0.3354, 

## 2. Fit DBSCAN and inspect groups

In [3]:
final_model = DBSCAN(eps=best["eps"], min_samples=best["min_samples"], metric="precomputed")
labels = final_model.fit_predict(distance_matrix)

is_noise = labels == -1
cluster_labels = [label for label in sorted(set(labels.tolist())) if label != -1]
sizes = Counter(labels)

print(
    f"Clusters (excluding noise): {len(cluster_labels)} | "
    f"noise points: {sizes.get(-1, 0)} / {len(labels)} ({100 * sizes.get(-1, 0) / len(labels):.1f}%)"
)
print(f"Cluster sizes (largest first): {[size for label, size in sizes.most_common() if label != -1]}")

members_by_cluster = {
    label: [attributes[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in cluster_labels
}

for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
    if len(members) > 1:
        print(f"\nCluster {label} ({len(members)} attributes)")
        for member in members:
            print(f"  - {member}")

Clusters (excluding noise): 59 | noise points: 280 / 445 (62.9%)
Cluster sizes (largest first): [9, 6, 6, 6, 5, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Cluster 2 (9 attributes)
  - Windproof Front Panel
  - Windproof protection
  - Windproof softshell 3-layer panels
  - Windtex Softshell 3-layer fabric
  - Windproof front panel
  - Windproof softshell fabric
  - Windproof front panels
  - Deep winter windproof protection
  - Windproof

Cluster 15 (6 attributes)
  - Highly breathable
  - Breathable
  - Extremely breathable
  - Super breathable
  - Extremely breathable 40K (40,000 g/m²/24h)
  - Breathable fabric

Cluster 17 (6 attributes)
  - Two-way YKK® Vislon zip
  - YKK 2-way zipper
  - Two-way YKK® Vislon AquaGuard zip
  - 2-way YKK front zipper
  - YKK Vislon 2-way zipper
  - YKK 2-way waterproof zipper

Cluster 33 (6 attributes)
  - Convenient rear pockets
  -

## 3. Cluster visualization

t-SNE projects precomputed distances (`1 - similarity`) to two dimensions for display only.
The largest groups receive highlight colors, smaller groups are grey, and noise points are hollow markers.

In [4]:
INK = "#2b2b2b"
GRAY = "#d7d7d7"
LIGHT_GRAY = "#c9c8c0"
HIGHLIGHT_PALETTE = ["#cc1f1a", "#005f73", "#0a9396", "#ee9b00", "#ae2012", "#6c757d", "#9b2226", "#3a86ff"]

perplexity = min(30, len(attributes) - 1)
coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    random_state=SEED,
    perplexity=perplexity,
).fit_transform(distance_matrix)

multi = [label for label in cluster_labels if sizes[label] > 1]
ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]

def group_name(label):
    indices = np.where(labels == label)[0]
    within_group_similarity = similarity[np.ix_(indices, indices)]
    medoid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return attributes[medoid_index]

group_names = {label: group_name(label) for label in multi}

def make_trace(indices, name, color, custom_group_names, hollow=False, size=9):
    if len(indices) == 0:
        return None
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[attributes[index] for index in indices],
        customdata=[custom_group_names[index] for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )

custom_group_names = []
for label in labels:
    if label == -1:
        custom_group_names.append("Noise")
    else:
        custom_group_names.append(group_names.get(label, f"Cluster {label}"))

noise_indices = np.where(labels == -1)[0]
other_group_indices = np.array(
    [index for index, label in enumerate(labels) if label != -1 and sizes[label] > 1 and label not in highlighted_groups],
    dtype=int,
)

figure = go.Figure()

trace = make_trace(noise_indices, f"Noise ({len(noise_indices)})", LIGHT_GRAY, custom_group_names, hollow=True, size=7)
if trace is not None:
    figure.add_trace(trace)

trace = make_trace(
    other_group_indices,
    f"Other groups ({max(len(ranked_groups) - len(highlighted_groups), 0)})",
    GRAY,
    custom_group_names,
    size=7,
)
if trace is not None:
    figure.add_trace(trace)

for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    trace = make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, custom_group_names, size=10)
    if trace is not None:
        figure.add_trace(trace)

figure.update_layout(
    title=dict(text="Attribute clusters: DBSCAN (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
try:
    figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))

## 5. Evaluation

### 5.1 Internal quality

DBSCAN marks some points as noise (`-1`). The selection score treats each noise point as its own singleton cluster (silhouette 0), so discarding attributes as noise does not inflate it. The silhouette on non-noise points only is shown for reference.

In [5]:
metrics = evaluate(labels, distance_matrix)

if metrics is None:
    print("Silhouette score not defined (need at least 2 non-noise clusters).")
else:
    print(f"Silhouette score (noise as singletons, selection score): {metrics['score']:.4f}")
    print(f"Silhouette score (non-noise only, reported only): {metrics['silhouette_non_noise']:.4f}")
    print(f"Clusters (excluding noise): {metrics['n_clusters']}")
    print(f"Coverage: {100 * metrics['coverage']:.1f}%")
    print(f"Noise points: {metrics['n_noise']} / {len(labels)}")

Silhouette score (noise as singletons, selection score): 0.1380
Silhouette score (non-noise only, reported only): 0.4755
Clusters (excluding noise): 59
Coverage: 37.1%
Noise points: 280 / 445


### 5.2 Parameter selection

These plots show how sensitive the result is to the chosen parameters, using the results already computed in the parameter search (no refitting).

- **k-distance plot** sorts every attribute's distance to its `min_samples`-th nearest neighbour (excluding itself). Points left of where the curve bends sharply upward lie in dense regions; the dashed line is the selected `eps`. If the line sits near the bend, `eps` separates dense attributes from sparse ones; attributes above the line cannot be core points.
- **Heatmaps** show the selection score, coverage and number of clusters for every configuration in the grid, with the same colour map in both notebooks. The red square marks the selected configuration. Blank cells produced fewer than 2 clusters. A selected cell inside a broad bright area means the choice is stable; an isolated peak, or one at the grid edge, means it depends on the exact values.
- **Sensitivity summary** lists the score and number of clusters one grid step away from the selected configuration in each direction. Small changes in score but large changes in cluster count mean the score alone does not pin down the granularity.

_Placeholder: DBSCAN finds N clusters with the selected parameters, compared with k = … chosen for spectral clustering in `5_spectral.ipynb`._

In [6]:
m = best["min_samples"]
nth = f"{m}{ {1: 'st', 2: 'nd', 3: 'rd'}.get(m if m < 20 else m % 10, 'th') }"
# Column 0 of each sorted row is the point itself (distance 0), so column m is the m-th neighbour.
k_distances = np.sort(np.sort(distance_matrix, axis=1)[:, m])

figure = go.Figure(
    go.Scatter(
        x=np.arange(1, len(k_distances) + 1), y=k_distances, mode="lines",
        line=dict(color="#2b2b2b", width=2), name=f"distance to {nth} neighbour",
        hovertemplate="rank %{x}<br>distance %{y:.3f}<extra></extra>",
    )
)
figure.add_hline(
    y=best["eps"], line=dict(color="#cc1f1a", dash="dash", width=2), opacity=1,
    annotation_text=f"selected eps = {best['eps']:.2f}", annotation_position="top left",
)
figure.update_layout(
    title=dict(text=f"k-distance plot (m = min_samples = {m})", x=0),
    template="simple_white", width=1000, height=450,
    xaxis_title="Attributes, sorted by distance", yaxis_title=f"Distance to {nth} nearest neighbour",
)
try:
    figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))

print(f"{(k_distances <= best['eps']).mean():.1%} of attributes have their {nth} neighbour within eps = {best['eps']:.2f}")

14.8% of attributes have their 2nd neighbour within eps = 0.40


In [7]:
SELECTED_COLOR = "#cc1f1a"
HEATMAP_COLORSCALE = "Viridis"
HEATMAP_METRICS = {
    "score": ("Score (silhouette, noise as singletons)", ".3f"),
    "coverage": ("Coverage (%)", ".1f"),
    "n_clusters": ("Clusters", ".0f"),
}


def grid_table(candidates, x_name, x_values, y_name, y_values):
    lookup = {(row[x_name], row[y_name]): row for row in candidates}
    table = {metric: np.full((len(y_values), len(x_values)), np.nan) for metric in HEATMAP_METRICS}
    for j, x in enumerate(x_values):
        for i, y in enumerate(y_values):
            row = lookup.get((x, y))
            if row is None:
                continue
            for metric in HEATMAP_METRICS:
                table[metric][i, j] = 100 * row[metric] if metric == "coverage" else row[metric]
    return table


def plot_param_heatmaps(table, x_name, x_values, y_name, y_values, best, method):
    x_labels = [f"{x:g}" for x in x_values]
    y_labels = [f"{y:g}" for y in y_values]
    figure = make_subplots(
        rows=len(HEATMAP_METRICS), cols=1,
        subplot_titles=[title for title, _ in HEATMAP_METRICS.values()],
        vertical_spacing=0.1,
    )
    for row_index, (metric, (title, fmt)) in enumerate(HEATMAP_METRICS.items(), start=1):
        domain = figure.get_subplot(row_index, 1).yaxis.domain
        figure.add_trace(
            go.Heatmap(
                z=table[metric], x=x_labels, y=y_labels,
                colorscale=HEATMAP_COLORSCALE, hoverongaps=False,
                colorbar=dict(len=domain[1] - domain[0], y=(domain[0] + domain[1]) / 2, thickness=12),
                hovertemplate=f"{x_name}=%{{x}}<br>{y_name}=%{{y}}<br>{title}: %{{z:{fmt}}}<extra></extra>",
            ),
            row=row_index, col=1,
        )
        figure.add_trace(
            go.Scatter(
                x=[f"{best[x_name]:g}"], y=[f"{best[y_name]:g}"], mode="markers",
                marker=dict(symbol="square-open", size=18, color=SELECTED_COLOR, line=dict(width=3)),
                name="Selected", showlegend=row_index == 1, hoverinfo="skip",
            ),
            row=row_index, col=1,
        )
        figure.update_xaxes(
            title_text=x_name if row_index == len(HEATMAP_METRICS) else None,
            type="category", tickangle=-45 if len(x_values) > 12 else 0, row=row_index, col=1,
        )
        figure.update_yaxes(title_text=y_name, type="category", row=row_index, col=1)
    figure.update_layout(
        title=dict(text=f"{method} parameter grid (blank = fewer than 2 clusters or not evaluated)", x=0),
        template="simple_white", width=1000, height=1000,
        legend=dict(orientation="h", x=1, xanchor="right", y=1.04),
    )
    show(figure)


def print_sensitivity(candidates, best, grids):
    names = list(grids)
    lookup = {tuple(row[name] for name in names): row for row in candidates}
    print(f"Selected: {format_row(best, grids)}")
    print("\nOne grid step away (score change vs selected):")
    for name, grid in grids.items():
        grid = list(grid)
        index = int(np.argmin([abs(value - best[name]) for value in grid]))
        for step in (-1, 1):
            if not 0 <= index + step < len(grid):
                print(f"  {name}: no {'lower' if step < 0 else 'higher'} value in grid")
                continue
            key = tuple(grid[index + step] if other == name else best[other] for other in names)
            params = ", ".join(f"{other}={value:g}" for other, value in zip(names, key))
            row = lookup.get(key)
            if row is None:
                print(f"  {params} | fewer than 2 clusters or not evaluated")
            else:
                print(
                    f"  {params} | score={row['score']:.4f} ({row['score'] - best['score']:+.4f}), "
                    f"clusters={row['n_clusters']}"
                )


def show(figure):
    try:
        figure.show(renderer="notebook_connected")
    except ValueError:
        display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))


grids = {"eps": EPS_VALUES.tolist(), "min_samples": MIN_SAMPLES_VALUES}
table = grid_table(candidates, "eps", grids["eps"], "min_samples", grids["min_samples"])
plot_param_heatmaps(table, "eps", grids["eps"], "min_samples", grids["min_samples"], best, "DBSCAN")
print_sensitivity(candidates, best, grids)

Selected: eps=0.4, min_samples=2 | score=0.1380, silhouette_non_noise=0.4755, clusters=59, coverage=37.1%, noise=280

One grid step away (score change vs selected):
  eps=0.38, min_samples=2 | score=0.1323 (-0.0058), clusters=56
  eps=0.42, min_samples=2 | score=0.1265 (-0.0115), clusters=61
  min_samples: no lower value in grid
  eps=0.4, min_samples=3 | score=0.0622 (-0.0759), clusters=24


## 4.3 HDBSCAN

# HDBSCAN clustering

Clusters all feature titles using the shared similarity matrix from `3_similarity.ipynb`. This follows the same workflow as `6_dbscan.ipynb`, but HDBSCAN can find clusters with different densities and marks outliers as noise.

**Method:** convert similarity to distance, compare a small grid of HDBSCAN parameters, inspect the selected clusters, select by silhouette with noise points counted as singleton clusters, visualize with t-SNE, and check parameter sensitivity with grid heatmaps.

**Requirement:** HDBSCAN is included in scikit-learn 1.3 and later.

In [1]:
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import HDBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

MIN_CLUSTER_SIZE_VALUES = [2, 3, 4, 5, 8, 12, 20]
MIN_SAMPLES_VALUES = [2, 3, 4, 5, 6, 8]
SEED = 0
HIGHLIGHT_GROUPS = 8

with np.load("data/similarity.npz") as data:
    similarity = data["similarity"]
    attributes = data["titles"].tolist()

n = len(attributes)
assert similarity.shape == (n, n)
assert np.allclose(similarity, similarity.T), "similarity must be symmetric"

distance_matrix = np.clip(1 - similarity, 0, 1)
np.fill_diagonal(distance_matrix, 0)

print(f"Loaded {n} attributes, matrix {similarity.shape}")

Loaded 445 attributes, matrix (445, 445)


## 1. Parameter search

`min_cluster_size` sets the smallest group HDBSCAN may return. `min_samples` controls how conservatively it treats points as noise; larger values usually label more points as noise.

Parameters are selected by the silhouette score computed on all attributes, with each noise point treated as its own singleton cluster (silhouette 0). This prevents a configuration from scoring well by discarding difficult attributes as noise. Ties are broken by higher coverage, then fewer clusters. Both methods search the same `min_samples` range; DBSCAN additionally searches `eps` and HDBSCAN `min_cluster_size`. HDBSCAN has no `eps` parameter.

The silhouette on non-noise points only is reported for reference but is not used for selection.

In [2]:
def noise_as_singletons(labels):
    relabeled = labels.copy()
    noise_indices = np.where(relabeled == -1)[0]
    start = relabeled.max() + 1
    relabeled[noise_indices] = np.arange(start, start + len(noise_indices))
    return relabeled


def evaluate(labels, D):
    is_noise = labels == -1
    n_noise = int(is_noise.sum())
    n_clusters = len(set(labels[~is_noise].tolist()))
    if n_clusters < 2:
        return None

    singleton_labels = noise_as_singletons(labels)
    if len(set(singleton_labels.tolist())) >= len(labels):
        return None
    score = silhouette_score(D, singleton_labels, metric="precomputed")

    non_noise_labels = labels[~is_noise]
    if n_clusters < len(non_noise_labels):
        silhouette_non_noise = silhouette_score(
            D[np.ix_(~is_noise, ~is_noise)], non_noise_labels, metric="precomputed"
        )
    else:
        silhouette_non_noise = float("nan")

    return {
        "score": float(score),
        "silhouette_non_noise": float(silhouette_non_noise),
        "coverage": float((~is_noise).mean()),
        "n_clusters": int(n_clusters),
        "n_noise": n_noise,
    }


def format_row(row, param_names):
    params = ", ".join(f"{name}={row[name]:g}" for name in param_names)
    return (
        f"{params} | "
        f"score={row['score']:.4f}, silhouette_non_noise={row['silhouette_non_noise']:.4f}, "
        f"clusters={row['n_clusters']}, coverage={100 * row['coverage']:.1f}%, noise={row['n_noise']}"
    )


def warn_if_at_grid_edge(best, grids):
    for name, grid in grids.items():
        if np.isclose(best[name], min(grid)) or np.isclose(best[name], max(grid)):
            print(f"Warning ({name}={best[name]:g}): Selected value is at the edge of the grid - consider extending the grid.")


PARAM_GRIDS = {"min_cluster_size": MIN_CLUSTER_SIZE_VALUES, "min_samples": MIN_SAMPLES_VALUES}

candidates = []
for min_cluster_size in MIN_CLUSTER_SIZE_VALUES:
    for min_samples in MIN_SAMPLES_VALUES:
        if min_samples > min_cluster_size:
            continue

        model = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="precomputed",
            copy=True,
        )
        labels = model.fit_predict(distance_matrix)
        evaluation = evaluate(labels, distance_matrix)
        if evaluation is None:
            continue
        candidates.append({"min_cluster_size": min_cluster_size, "min_samples": min_samples, **evaluation})

if not candidates:
    raise ValueError("No valid HDBSCAN configuration produced at least 2 non-noise clusters.")

candidates = sorted(candidates, key=lambda r: (-r["score"], -r["coverage"], r["n_clusters"]))
best = candidates[0]

print("Top 10 parameter sets (score = silhouette with noise as singletons):")
for row in candidates[:10]:
    print(format_row(row, PARAM_GRIDS))

print("\nSelected configuration:")
print(format_row(best, PARAM_GRIDS))
warn_if_at_grid_edge(best, PARAM_GRIDS)

Top 10 parameter sets (score = silhouette with noise as singletons):
min_cluster_size=2, min_samples=2 | score=0.1778, silhouette_non_noise=0.2813, clusters=106, coverage=73.3%, noise=119
min_cluster_size=3, min_samples=2 | score=0.1103, silhouette_non_noise=0.2360, clusters=61, coverage=67.0%, noise=147
min_cluster_size=4, min_samples=2 | score=0.0725, silhouette_non_noise=0.2292, clusters=39, coverage=59.6%, noise=180
min_cluster_size=3, min_samples=3 | score=0.0520, silhouette_non_noise=0.2924, clusters=38, coverage=47.0%, noise=236
min_cluster_size=5, min_samples=2 | score=0.0473, silhouette_non_noise=0.2218, clusters=30, coverage=56.9%, noise=192
min_cluster_size=4, min_samples=4 | score=0.0149, silhouette_non_noise=0.2833, clusters=20, coverage=41.1%, noise=262
min_cluster_size=5, min_samples=4 | score=0.0147, silhouette_non_noise=0.2814, clusters=18, coverage=39.3%, noise=270
min_cluster_size=4, min_samples=3 | score=0.0140, silhouette_non_noise=0.2430, clusters=25, coverage=47.

## 2. Fit HDBSCAN and inspect groups

In [3]:
final_model = HDBSCAN(
    min_cluster_size=best["min_cluster_size"],
    min_samples=best["min_samples"],
    metric="precomputed",
    copy=True,
)
labels = final_model.fit_predict(distance_matrix)

sizes = Counter(labels.tolist())
cluster_labels = [label for label in sorted(sizes) if label != -1]
n_noise = sizes.get(-1, 0)

print(
    f"Clusters (excluding noise): {len(cluster_labels)} | "
    f"coverage: {100 * (len(labels) - n_noise) / len(labels):.1f}% | "
    f"noise points: {n_noise} / {len(labels)} ({100 * n_noise / len(labels):.1f}%)"
)
print(f"Cluster sizes (largest first): {[sizes[label] for label in sorted(cluster_labels, key=lambda label: -sizes[label])]}")

members_by_cluster = {
    label: [attributes[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in cluster_labels
}

for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
    print(f"\nCluster {label} ({len(members)} attributes)")
    for member in members:
        print(f"  - {member}")

Clusters (excluding noise): 106 | coverage: 73.3% | noise points: 119 / 445 (26.7%)
Cluster sizes (largest first): [9, 8, 7, 7, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Cluster 105 (9 attributes)
  - Windproof Front Panel
  - Windproof protection
  - Windproof softshell 3-layer panels
  - Windtex Softshell 3-layer fabric
  - Windproof front panel
  - Windproof softshell fabric
  - Windproof front panels
  - Deep winter windproof protection
  - Windproof

Cluster 88 (8 attributes)
  - Merino wool fabric mix
  - Luxurious soft merino blend
  - Merino wool lining
  - Pure merino wool
  - Merino wool fabric blend
  - Durable Merino-Polyester Blend
  - Merino wool blend
  - Premium merino wool blend

Cluster 29 (7 attributes)
  -

## 3. Cluster visualization

t-SNE projects the precomputed distances (`1 - similarity`) to two dimensions for display only. The largest clusters get highlight colors, smaller clusters are grey, and noise points are shown as hollow markers.

In [4]:
coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    learning_rate="auto",
    random_state=SEED,
).fit_transform(distance_matrix)

PALETTE = ["#2f6b8a", "#e07a5f", "#3d9970", "#8e6cbb", "#d4a72c", "#d45d79", "#5086c1", "#8c6d5c"]
ranked_clusters = sorted(cluster_labels, key=lambda label: (-sizes[label], label))
highlighted_clusters = ranked_clusters[:HIGHLIGHT_GROUPS]
group_names = {label: f"Cluster {label}" for label in cluster_labels}

def add_points(indices, name, color, hollow=False, size=8):
    if len(indices) == 0:
        return

    marker = {"size": size, "color": color}
    if hollow:
        marker.update({"symbol": "circle-open", "line": {"color": color, "width": 1.5}})

    figure.add_trace(
        go.Scatter(
            x=coordinates[indices, 0],
            y=coordinates[indices, 1],
            mode="markers",
            name=name,
            marker=marker,
            text=[attributes[index] for index in indices],
            customdata=["Noise" if labels[index] == -1 else group_names[labels[index]] for index in indices],
            hovertemplate="<b>%{text}</b><br>%{customdata}<extra></extra>",
        )
    )

figure = go.Figure()
noise_indices = np.where(labels == -1)[0]
add_points(noise_indices, f"Noise ({len(noise_indices)})", "#b8bec5", hollow=True, size=7)

other_indices = np.array(
    [index for index, label in enumerate(labels) if label != -1 and label not in highlighted_clusters],
    dtype=int,
)
add_points(other_indices, "Other clusters", "#b8bec5", size=7)

for color, label in zip(PALETTE, highlighted_clusters):
    indices = np.where(labels == label)[0]
    add_points(indices, f"Cluster {label} ({sizes[label]})", color, size=10)

figure.update_layout(
    title=dict(text="Attribute clusters: HDBSCAN (t-SNE)", x=0),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", itemsizing="constant"),
    hoverlabel=dict(bgcolor="white"),
)
try:
    figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))

## 5. Evaluation

### 5.1 Internal quality

HDBSCAN marks some points as noise (`-1`). The selection score treats each noise point as its own singleton cluster (silhouette 0), so discarding attributes as noise does not inflate it. The silhouette on non-noise points only is shown for reference.

In [5]:
metrics = evaluate(labels, distance_matrix)

if metrics is None:
    print("Silhouette score not defined (need at least 2 non-noise clusters).")
else:
    print(f"Silhouette score (noise as singletons, selection score): {metrics['score']:.4f}")
    print(f"Silhouette score (non-noise only, reported only): {metrics['silhouette_non_noise']:.4f}")
    print(f"Clusters (excluding noise): {metrics['n_clusters']}")
    print(f"Coverage: {100 * metrics['coverage']:.1f}%")
    print(f"Noise points: {metrics['n_noise']} / {len(labels)}")

Silhouette score (noise as singletons, selection score): 0.1778
Silhouette score (non-noise only, reported only): 0.2813
Clusters (excluding noise): 106
Coverage: 73.3%
Noise points: 119 / 445


### 5.2 Parameter selection

These plots show how sensitive the result is to the chosen parameters, using the results already computed in the parameter search (no refitting).

- **Heatmaps** show the selection score, coverage and number of clusters for every configuration in the grid, with the same colour map in both notebooks. The red square marks the selected configuration. Blank cells produced fewer than 2 clusters or were skipped because `min_samples > min_cluster_size`. A selected cell inside a broad bright area means the choice is stable; an isolated peak, or one at the grid edge, means it depends on the exact values.
- **Sensitivity summary** lists the score and number of clusters one grid step away from the selected configuration in each direction. Small changes in score but large changes in cluster count mean the score alone does not pin down the granularity.

_Placeholder: HDBSCAN finds N clusters with the selected parameters, compared with k = … chosen for spectral clustering in `5_spectral.ipynb`._

In [6]:
SELECTED_COLOR = "#cc1f1a"
HEATMAP_COLORSCALE = "Viridis"
HEATMAP_METRICS = {
    "score": ("Score (silhouette, noise as singletons)", ".3f"),
    "coverage": ("Coverage (%)", ".1f"),
    "n_clusters": ("Clusters", ".0f"),
}


def grid_table(candidates, x_name, x_values, y_name, y_values):
    lookup = {(row[x_name], row[y_name]): row for row in candidates}
    table = {metric: np.full((len(y_values), len(x_values)), np.nan) for metric in HEATMAP_METRICS}
    for j, x in enumerate(x_values):
        for i, y in enumerate(y_values):
            row = lookup.get((x, y))
            if row is None:
                continue
            for metric in HEATMAP_METRICS:
                table[metric][i, j] = 100 * row[metric] if metric == "coverage" else row[metric]
    return table


def plot_param_heatmaps(table, x_name, x_values, y_name, y_values, best, method):
    x_labels = [f"{x:g}" for x in x_values]
    y_labels = [f"{y:g}" for y in y_values]
    figure = make_subplots(
        rows=len(HEATMAP_METRICS), cols=1,
        subplot_titles=[title for title, _ in HEATMAP_METRICS.values()],
        vertical_spacing=0.1,
    )
    for row_index, (metric, (title, fmt)) in enumerate(HEATMAP_METRICS.items(), start=1):
        domain = figure.get_subplot(row_index, 1).yaxis.domain
        figure.add_trace(
            go.Heatmap(
                z=table[metric], x=x_labels, y=y_labels,
                colorscale=HEATMAP_COLORSCALE, hoverongaps=False,
                colorbar=dict(len=domain[1] - domain[0], y=(domain[0] + domain[1]) / 2, thickness=12),
                hovertemplate=f"{x_name}=%{{x}}<br>{y_name}=%{{y}}<br>{title}: %{{z:{fmt}}}<extra></extra>",
            ),
            row=row_index, col=1,
        )
        figure.add_trace(
            go.Scatter(
                x=[f"{best[x_name]:g}"], y=[f"{best[y_name]:g}"], mode="markers",
                marker=dict(symbol="square-open", size=18, color=SELECTED_COLOR, line=dict(width=3)),
                name="Selected", showlegend=row_index == 1, hoverinfo="skip",
            ),
            row=row_index, col=1,
        )
        figure.update_xaxes(
            title_text=x_name if row_index == len(HEATMAP_METRICS) else None,
            type="category", tickangle=-45 if len(x_values) > 12 else 0, row=row_index, col=1,
        )
        figure.update_yaxes(title_text=y_name, type="category", row=row_index, col=1)
    figure.update_layout(
        title=dict(text=f"{method} parameter grid (blank = fewer than 2 clusters or not evaluated)", x=0),
        template="simple_white", width=1000, height=1000,
        legend=dict(orientation="h", x=1, xanchor="right", y=1.04),
    )
    show(figure)


def print_sensitivity(candidates, best, grids):
    names = list(grids)
    lookup = {tuple(row[name] for name in names): row for row in candidates}
    print(f"Selected: {format_row(best, grids)}")
    print("\nOne grid step away (score change vs selected):")
    for name, grid in grids.items():
        grid = list(grid)
        index = int(np.argmin([abs(value - best[name]) for value in grid]))
        for step in (-1, 1):
            if not 0 <= index + step < len(grid):
                print(f"  {name}: no {'lower' if step < 0 else 'higher'} value in grid")
                continue
            key = tuple(grid[index + step] if other == name else best[other] for other in names)
            params = ", ".join(f"{other}={value:g}" for other, value in zip(names, key))
            row = lookup.get(key)
            if row is None:
                print(f"  {params} | fewer than 2 clusters or not evaluated")
            else:
                print(
                    f"  {params} | score={row['score']:.4f} ({row['score'] - best['score']:+.4f}), "
                    f"clusters={row['n_clusters']}"
                )


def show(figure):
    try:
        figure.show(renderer="notebook_connected")
    except ValueError:
        display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))


grids = {"min_cluster_size": MIN_CLUSTER_SIZE_VALUES, "min_samples": MIN_SAMPLES_VALUES}
table = grid_table(candidates, "min_cluster_size", grids["min_cluster_size"], "min_samples", grids["min_samples"])
plot_param_heatmaps(table, "min_cluster_size", grids["min_cluster_size"], "min_samples", grids["min_samples"], best, "HDBSCAN")
print_sensitivity(candidates, best, grids)

Selected: min_cluster_size=2, min_samples=2 | score=0.1778, silhouette_non_noise=0.2813, clusters=106, coverage=73.3%, noise=119

One grid step away (score change vs selected):
  min_cluster_size: no lower value in grid
  min_cluster_size=3, min_samples=2 | score=0.1103 (-0.0675), clusters=61
  min_samples: no lower value in grid
  min_cluster_size=2, min_samples=3 | fewer than 2 clusters or not evaluated


# 5. Graph clustering

## 5.1 General spectral clustering

# Generalized spectral clustering

Clusters all 445 feature names together, without filtering by product type. 

Our aim is to find groups of feature names that are strongly linked inside and only weakly linked to the rest.

We start from the shared similarity matrix: `3_similarity.ipynb` now saves it to `data/similarity.npz`, and this notebook (like the agglomerative and DBSCAN ones) just loads it.

**Method** (normalized spectral clustering):
1. We build a graph from the similarity matrix: feature names are nodes, similarities are edge weights (each node keeps its `K_NEIGHBORS` most similar neighbours).
2. We compute the normalized Laplacian and its eigenvectors with the smallest eigenvalues.
3. Finally, we run k-means on those eigenvectors, with k = 35 clusters (chosen in section 5.2).

In [1]:
import json
import time
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import KMeans
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import davies_bouldin_score, silhouette_score

#links kept per feature name
K_NEIGHBORS = 15

#number of clusters, chosen from the k sweep and the clusters themselves (see 5.2)
K_CLUSTERS = 35

#to make random parts repeatable
SEED = 0

#loading the similarity matrix (showing how alike each pair of feature names is)
with np.load("data/similarity.npz") as data:
    S = data["similarity"]
    #list of the feature names in the same order as in the matrix
    feature_names = data["titles"].tolist()


#preliminary checks on the similarity matrix
n = len(feature_names)
assert S.shape == (n, n), f"matrix {S.shape} does not match {n} feature names"
assert len(set(feature_names)) == n, "duplicate feature names"
assert S.min() >= 0 and S.max() <= 1 + 1e-9, "values are not in the interval [0,1]"
assert np.allclose(S, S.T), "similarity must be symmetric"

print(f"Loaded {n} feature names and matrix {S.shape}")

Loaded 445 feature names and matrix (445, 445)


## Build the similarity graph

The diagonal (originally set as 1, since all feature names are obviously strongly similar (identical!) to themselves) is set to 0 (we want no self-loops to W and that a node's own entry doesn't take one of its k neighbour slots). 

With `K_NEIGHBORS`, each feature name keeps only its most similar neighbours; the result is symmetrised so that an edge is kept if either endpoint selects it. 

This makes the graph sparse, removes weak noisy links, and is what makes spectral clustering scale to larger catalogs.

In [2]:
def build_affinity(S, k_neighbors=K_NEIGHBORS):
    W = S.copy()
    np.fill_diagonal(W, 0)

    #sorts each row from smallest to largest, take k-th value from the end of row, turn values into a column
    threshold = np.sort(W, axis=1)[:, -k_neighbors][:, None]

    #we check each pair (x,y): if x is in y´s top K OR viceversa, the edge is kept
    #otherwise it is set to 0 
    W = np.where((W >= threshold) | (W >= threshold.T), W, 0) 
    return W


W = build_affinity(S, K_NEIGHBORS)

#check W is symmetric
assert np.allclose(W, W.T)

#weighted degree of d_i: sum of the similarities on node i's edges
deg = W.sum(axis=1)
#how many neighbors each node has 
nbrs = (W > 0).sum(axis=1)
#number of connected components
n_comp, _ = connected_components(W > 0, directed=False)

print(f"Edges kept: {np.count_nonzero(np.triu(W))} of {n*(n-1)//2} ({np.count_nonzero(np.triu(W))/(n*(n-1)//2):.1%})")

print(f"Neighbours per node: min {nbrs.min()}, median {int(np.median(nbrs))}, max {nbrs.max()}")

print(f"Weighted degree: min {deg.min():.2f}, median {np.median(deg):.2f}, max {deg.max():.2f}")

print(f"Isolated nodes: {(deg == 0).sum()}, connected components: {n_comp}")

Edges kept: 4618 of 98790 (4.7%)
Neighbours per node: min 15, median 18, max 59
Weighted degree: min 1.01, median 5.60, max 18.85
Isolated nodes: 0, connected components: 1


## Normalized Laplacian and eigengap

`np.linalg.eigh` is used because $L_{sym}$ is symmetric; it returns eigenvalues in ascending order. Small eigenvalues correspond to well-separated groups; the largest jump (eigengap) among them suggests the number of clusters. We report the eigengap's suggestion for reference, but set k from the sweep in section 5.2.

In [3]:
def normalized_laplacian(W):
    # matrix whose eigenvectors give the best ways to split the graph into groups
    #L_sym = I - D^(-1/2) W D^(-1/2): each link w_ij is rescaled to w_ij / sqrt(d_i * d_j)
    
    #without it, very connected feature names would dominate and pull everything towards them
    #after rescaling, a link counts more when it’s important to both sides
    
    deg = W.sum(axis=1)   

    #computing 1/sqrt(d) for each node                       
    d_inv_sqrt = 1 / np.sqrt(np.maximum(deg, 1e-12)) # 1e-12 avoids division by zero
    #eye build the identity matrix, outer builds a multiplication table having d_inv_sqrt as both a row and a column
    #w is multiplied element wise
    return np.eye(len(W)) - W * np.outer(d_inv_sqrt, d_inv_sqrt)


def spectral_embedding(W, n_eigen):
    """Return the n_eigen smallest eigenvalues of L_sym and their eigenvectors."""
    eigenvalues, eigenvectors = np.linalg.eigh(normalized_laplacian(W))
    return eigenvalues[:n_eigen], eigenvectors[:, :n_eigen]

# eigengap search range, reported for reference only (k is set in the first cell, see 5.2)
K_MIN, K_MAX = 20, 80

eigenvalues, eigenvectors = spectral_embedding(W, n_eigen=K_MAX + 1)

gaps = np.diff(eigenvalues)

# number of small eigenvalues before the biggest jump = number of clusters
k_eigengap = int(np.argmax(gaps[K_MIN - 1:K_MAX]) + K_MIN)

print(f"Eigengap between k={K_MIN} and {K_MAX} suggests k = {k_eigengap} "
      f"(gap = {gaps[k_eigengap - 1]:.4f}); we use k = {K_CLUSTERS} (see 5.2)")

Eigengap between k=20 and 80 suggests k = 25 (gap = 0.0286); we use k = 35 (see 5.2)


## 4. Spectral clustering

Rows of the first *k* eigenvectors are normalized to unit length (Ng–Jordan–Weiss step), then clustered with k-means.

In [4]:
def spectral_cluster(eigenvectors, k, seed=SEED):
    U = eigenvectors[:, :k]
    U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
    labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(U)
    return labels, U


k = K_CLUSTERS
labels, U = spectral_cluster(eigenvectors, k)
sizes = Counter(labels)
print(f"{k} clusters; sizes (largest first): {[size for _, size in sizes.most_common()]}")


cluster_sizes = Counter(labels.tolist())
cluster_labels = sorted(cluster_sizes)
singleton_clusters = [cluster_label for cluster_label in cluster_labels if cluster_sizes[cluster_label] == 1]

print(
    f"Clusters: {len(cluster_labels)} | "
    f"singleton clusters: {len(singleton_clusters)} / {len(feature_names)} feature names "
    f"({100 * len(singleton_clusters) / len(feature_names):.1f}%)"
)
print(f"Cluster sizes (largest first): {[size for _, size in cluster_sizes.most_common()]}")

feature_names_by_cluster = {
    cluster_label: [
        feature_names[index]
        for index, assigned_cluster_label in enumerate(labels)
        if assigned_cluster_label == cluster_label
    ]
    for cluster_label in cluster_labels
}

for cluster_label, cluster_feature_names in sorted(
    feature_names_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])
):
    if len(cluster_feature_names) > 1:
        print(f"\nCluster {cluster_label} ({len(cluster_feature_names)} feature names)")
        for feature_name in cluster_feature_names:
            print(f"  - {feature_name}")

35 clusters; sizes (largest first): [24, 22, 18, 17, 17, 17, 16, 16, 16, 15, 14, 14, 14, 13, 13, 12, 12, 12, 12, 11, 11, 11, 11, 10, 10, 10, 10, 10, 9, 9, 8, 8, 8, 8, 7]
Clusters: 35 | singleton clusters: 0 / 445 feature names (0.0%)
Cluster sizes (largest first): [24, 22, 18, 17, 17, 17, 16, 16, 16, 15, 14, 14, 14, 13, 13, 12, 12, 12, 12, 11, 11, 11, 11, 10, 10, 10, 10, 10, 9, 9, 8, 8, 8, 8, 7]

Cluster 1 (24 feature names)
  - Highly breathable
  - Breathable mesh back
  - Insulating & breathable
  - Open mesh zones
  - Breathable
  - Airflow mesh
  - Side mesh panels
  - Cool and comfortable
  - Breathable construction
  - Ventilating design
  - Extremely breathable
  - Super breathable
  - Ideal for hot conditions
  - Thermal insulation with exceptional breathability
  - Insulating & Breathable
  - Air flow
  - Breathable mesh inserts
  - Breathable insulated backhand
  - Extremely breathable 40K (40,000 g/m²/24h)
  - Maximum breathability
  - Breathable 15K
  - Breathable mesh bac

## 5. Evaluation

Do the clusters actually group feature names that are similar to each other? To answer it fairly, we evaluate every result on the original similarities `S` (or the distances `1 - S`). This is the common ground all methods start from: the agglomerative, DBSCAN and HDBSCAN notebooks receive exactly the same matrix and have no graph or embedding of their own. 

- 5.1 Internal quality (silhouette, Davies-Bouldin, modularity)
- 5.2 Choosing *k*
- 5.3 Coherence (to be completed)
- 5.4 Outliers, density and radius
- 5.5 Runtime
- 5.6 Stability and correctness

### 5.1 Internal quality

Here we check how good the clusters produced by spectral clustering are. We take the labels from section 4 and measure them against the shared similarity matrix `S`, which stays fixed.

The **silhouette** looks at single feature names. For each one, it asks whether the feature name is closer to the other members of its own cluster than to the nearest other cluster, with score going from −1 to 1.

The **Davies-Bouldin index** looks at pairs of clusters. For each cluster, it finds the neighbouring cluster it overlaps with the most, and compares how spread out the two clusters are with how far apart they are.

**Modularity** looks at the links. It compares the similarity inside the clusters with the similarity we would expect if links were spread at random.

In [5]:
from sklearn.metrics import silhouette_samples, adjusted_rand_score   # used in 5.4 and 5.6
from sklearn.cluster import SpectralClustering                        # used in 5.6

# Shared inputs
D = 1 - S
np.fill_diagonal(D, 0)
G = S.copy()       
np.fill_diagonal(G, 0)


def clustroid(labels):
    #to find each cluster's most central member
    result = {}
    for label in np.unique(labels):
        members = np.where(labels == label)[0]
        result[label] = members[D[np.ix_(members, members)].sum(axis=1).argmin()]
    return result


def silhouette(labels):
    return float(silhouette_score(D, labels, metric="precomputed"))


def davies_bouldin(labels):
    #find each cluster center
    centre = clustroid(labels)
    clusters = list(centre)
    #compute avg distance of members of a cluster to its clustroid
    spread = np.array([D[labels == c, centre[c]].mean() for c in clusters])
    centres = [centre[c] for c in clusters]

    separation = D[np.ix_(centres, centres)] + np.diag([np.inf] * len(clusters))
    #for every pair of clusters we compute (spread_i + spread_j) / distance between their centres
    ratios = (spread[:, None] + spread[None, :]) / separation
    #for each cluster we keep its worst ratio (its most overlapping neighbour)
    worst = ratios.max(axis=1)
    #computing average over all clusters
    return float(worst.mean())

def modularity(labels):
    #degree of each feature name
    degree = G.sum(axis=1)
    two_m = degree.sum()
    #true-false table to find pairs in the same cluster
    same = labels[:, None] == labels[None, :]
    return float(((G - np.outer(degree, degree) / two_m) * same).sum() / two_m)


def evaluate(labels):
    return {"silhouette": silhouette(labels),
            "davies_bouldin": davies_bouldin(labels),
            "modularity": modularity(labels)}


scores = evaluate(labels)
print(f"k = {k}")
for name, value in scores.items():
    print(f"  {name:15} {value:.3f}")

k = 35
  silhouette      0.137
  davies_bouldin  1.585
  modularity      0.104


### 5.2 Choosing *k*

We sweep *k* from 10 to 100, keeping the graph and the eigenvectors fixed, and plot the three measures plus the size of the largest and smallest cluster. The dashed grey line marks the eigengap suggestion and the solid black line our choice, k = 35. A good *k* has a high silhouette and modularity, a low Davies-Bouldin, and no oversized catch-all cluster.

**Choice of k = 35.** The eigengap suggests k = 25, but there the silhouette is still rising steeply. We chose k = 35 after inspecting this sweep and the clusters themselves: the silhouette has levelled off (about 0.14), no cluster has fewer than 7 feature names, and clusters that were mixed at k = 25 (for example DWR coating together with neoprene, windproof together with UPF protection, and a group of cut and cuff details) split into meaningful parts. Beyond k ≈ 40 the silhouette barely improves while clusters of only 2–4 feature names appear. Modularity decreases and Davies-Bouldin improves steadily with k, so neither of them points to a single best k. This choice was made after looking at the results, not by a fixed rule.

In [6]:
K_SWEEP = range(10, 101, 5)
_, eigenvectors_sweep = spectral_embedding(W, n_eigen=max(K_SWEEP))

# Cluster and evaluate for every k
sweep = []
for k_ in K_SWEEP:
    labels_k, _ = spectral_cluster(eigenvectors_sweep, k_)
    sizes_k = Counter(labels_k).values()
    sweep.append({"k": k_, **evaluate(labels_k), "largest": max(sizes_k), "smallest": min(sizes_k)})

# Table
print(f"{'k':>4} {'silhouette':>11} {'DB':>6} {'modularity':>11} {'largest':>8} {'smallest':>9}")
for r in sweep:
    print(f"{r['k']:>4} {r['silhouette']:>11.3f} {r['davies_bouldin']:>6.2f} "
          f"{r['modularity']:>11.3f} {r['largest']:>8} {r['smallest']:>9}")

# Plot
ks = [r["k"] for r in sweep]
panels = {"Silhouette ↑": ["silhouette"], "Davies-Bouldin ↓": ["davies_bouldin"],
          "Modularity ↑": ["modularity"], "Cluster size": ["largest", "smallest"]}

figure = make_subplots(rows=2, cols=2, subplot_titles=list(panels))
for i, metrics in enumerate(panels.values()):
    for metric in metrics:
        figure.add_trace(go.Scatter(x=ks, y=[r[metric] for r in sweep], mode="lines+markers", name=metric),
                         row=i // 2 + 1, col=i % 2 + 1)
figure.add_vline(x=k_eigengap, line_dash="dash", line_color="gray", row="all", col="all")
figure.add_vline(x=K_CLUSTERS, line_color="black", row="all", col="all")
figure.update_xaxes(title_text="k")
figure.update_layout(template="simple_white", width=1000, height=650, showlegend=False,
                     title="Spectral clustering: quality vs number of clusters")
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

   k  silhouette     DB  modularity  largest  smallest
  10       0.088   1.65       0.191      108        14
  15       0.101   1.69       0.164       73        10
  20       0.111   1.70       0.138       34        10
  25       0.122   1.65       0.124       31         9
  30       0.134   1.66       0.117       24         7
  35       0.137   1.59       0.104       24         7
  40       0.132   1.60       0.096       23         6
  45       0.146   1.57       0.090       16         4
  50       0.148   1.53       0.085       15         3
  55       0.135   1.54       0.077       15         3
  60       0.145   1.50       0.073       13         3
  65       0.136   1.58       0.070       13         3
  70       0.145   1.52       0.066       14         2
  75       0.145   1.44       0.063       14         2
  80       0.145   1.40       0.058       12         2
  85       0.139   1.43       0.055       11         1
  90       0.142   1.38       0.053       11         2
  95      

### 5.3 Coherence: does each cluster describe one property?

_TO BE COMPLETED_

Hand-labelled pairs or manual inspection?

### 5.4 Outliers, density and radius

Here we look inside each cluster. Its **density** is the average similarity between its members: the higher it is, the tighter the group. Its **radius** is the distance from the clustroid to the member furthest away, which shows how far the cluster stretches; we also report the average distance to the clustroid. Finally, we count the **outliers**: members with a negative silhouette, which are on average closer to another cluster than to their own.

In [7]:
silhouette_per_feature_name = silhouette_samples(D, labels, metric="precomputed")

# Index of each cluster's clustroid
centre = clustroid(labels)

#we show one row of statistics per cluster
rows = []
for label, size in sizes.most_common():
    members = np.where(labels == label)[0]
    within = S[np.ix_(members, members)]

    #density formula
    density = (within.sum() - size) / (size * (size - 1)) if size > 1 else 1.0   # mean off-diagonal similarity
    to_clustroid = D[members, centre[label]]
    rows.append({
        "cluster": label, "size": size, "density": density,
        "mean_dist": to_clustroid.mean(), "radius": to_clustroid.max(),
        "outliers": int((silhouette_per_feature_name[members] < 0).sum()),
        "clustroid": feature_names[centre[label]],
    })

print(f"{'cluster':>7} {'size':>5} {'density':>8} {'mean dist':>10} {'radius':>7} {'outliers':>9}  clustroid")
for row in rows:
    print(f"{row['cluster']:>7} {row['size']:>5} {row['density']:>8.3f} {row['mean_dist']:>10.3f} "
          f"{row['radius']:>7.3f} {row['outliers']:>9}  {row['clustroid']}")

print("\nQuartiles across clusters (25% / 50% / 75%):")
for metric in ["size", "density", "mean_dist", "radius"]:
    q = np.percentile([row[metric] for row in rows], [25, 50, 75])
    print(f"  {metric:10} {q[0]:.3f} / {q[1]:.3f} / {q[2]:.3f}")

outliers = np.where(silhouette_per_feature_name < 0)[0]
singletons = sum(1 for size in sizes.values() if size == 1)
print(f"\nOutliers (negative silhouette): {len(outliers)} of {n} ({len(outliers) / n:.1%}); singletons: {singletons}")

print("\nWorst 10 feature names (lowest silhouette):")
for i in np.argsort(silhouette_per_feature_name)[:10]:
    print(f"  {silhouette_per_feature_name[i]:6.3f}  {feature_names[i]!r}  in cluster '{feature_names[centre[labels[i]]]}'")

cluster  size  density  mean dist  radius  outliers  clustroid
      1    24    0.334      0.521   0.876         0  Highly breathable
      8    22    0.118      0.769   1.000        11  Rear hook-and-loop closure
      7    18    0.311      0.560   0.877         0  Silicone palm grip
     23    17    0.377      0.499   0.717         0  Deep winter insulation
     17    17    0.262      0.608   0.855         1  Durable and reliable
      9    17    0.197      0.677   0.939        13  Soft brushed lining
      4    16    0.383      0.491   0.765         0  Waterproof protection
      5    16    0.369      0.527   0.693         0  High stretch
     11    16    0.312      0.565   0.850         0  Rear pocket storage
     22    15    0.214      0.650   0.889         3  Active fit
     31    14    0.310      0.531   0.828         1  Moisture wicking & quick-drying
     33    14    0.277      0.585   0.785         0  Reinforced heel and toe
     15    14    0.223      0.659   0.854         0

### 5.5 Runtime

We timed the three stages of the pipeline (graph construction, eigendecomposition, k-means) on random subsets of increasing size, with the same `K_NEIGHBORS` and *k*. Each size is run 3 times and  we kept the median value

**Complexity:** _TO BE COMPLETED_

In [8]:
SUBSET_SIZES = [50, 100, 200, 300, n]  
REPEATS = 3                            
rng = np.random.default_rng(SEED)


def time_once(size):
    subset = rng.choice(n, size, replace=False)
    t0 = time.perf_counter()
    W_sub = build_affinity(S[np.ix_(subset, subset)], K_NEIGHBORS)  
    t1 = time.perf_counter()
    _, vectors = spectral_embedding(W_sub, n_eigen=k)               
    t2 = time.perf_counter()
    spectral_cluster(vectors, k)                                      
    t3 = time.perf_counter()
    return [t1 - t0, t2 - t1, t3 - t2]


stages = ["graph", "eigendecomposition", "k-means"]
# for each size we compute median over the repeats
timings = np.array([np.median([time_once(size) for _ in range(REPEATS)], axis=0) for size in SUBSET_SIZES])
total = timings.sum(axis=1)

# Table
print(f"{'n':>5} " + "".join(f"{s:>20}" for s in stages) + f"{'total':>10}")
for size, row, t in zip(SUBSET_SIZES, timings, total):
    print(f"{size:>5} " + "".join(f"{v:>19.4f}s" for v in row) + f"{t:>9.4f}s")

# Plot
figure = go.Figure()
for stage, values in zip(stages + ["total"], [*timings.T, total]):
    figure.add_trace(go.Scatter(x=SUBSET_SIZES, y=values, mode="lines+markers", name=stage,
                                line=dict(dash="dash" if stage == "total" else "solid")))
figure.update_layout(title="Spectral clustering runtime by stage", xaxis_title="number of feature names",
                     yaxis_title="seconds (median of 3)", template="simple_white", width=800, height=450)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

    n                graph  eigendecomposition             k-means     total
   50              0.0003s             0.0005s             0.0222s   0.0229s
  100              0.0002s             0.0009s             0.0257s   0.0268s
  200              0.0005s             0.0025s             0.0311s   0.0341s
  300              0.0011s             0.0070s             0.0282s   0.0363s
  445              0.0022s             0.0132s             0.0335s   0.0489s


## 6. Cluster visualization

t-SNE projects the same precomputed distances (`1 - similarity`) to two dimensions for display only; it does not change the clustering labels or serve as an evaluation metric. Nearby points suggest similarity, but the projection can distort some distances. The largest groups get distinct colours, the others are grey, and singletons (if any) are hollow markers. Hover over a point to see its feature name and group.

In [9]:
from sklearn.manifold import TSNE

HIGHLIGHT_GROUPS = 8
HIGHLIGHT_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY, INK, MUTED = "#8c8c86", "#c9c8c0", "#1f2933", "#6b7785"

distance_matrix = 1 - S
multi = [label for label, size in sizes.items() if size > 1]

coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    perplexity=30,
    random_state=42,
).fit_transform(distance_matrix)


def group_name(label):
    """Use the most centrally similar feature name as the group's display name."""
    indices = np.where(labels == label)[0]
    within_group_similarity = S[np.ix_(indices, indices)]
    clustroid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return feature_names[clustroid_index]


ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]
group_names = {label: group_name(label) for label in multi}


def make_trace(indices, name, color, hollow=False, size=9):
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[feature_names[index] for index in indices],
        customdata=[group_names.get(labels[index], "Singleton") for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )


other_group_indices = np.array(
    [index for index, label in enumerate(labels) if sizes[label] > 1 and label not in highlighted_groups],
    dtype=int,
)
singleton_indices = np.array([index for index, label in enumerate(labels) if sizes[label] == 1], dtype=int)

figure = go.Figure()
figure.add_trace(
    make_trace(singleton_indices, f"Singletons ({len(singleton_indices)})", LIGHT_GRAY, hollow=True, size=7)
)
figure.add_trace(
    make_trace(other_group_indices, f"Other groups ({len(ranked_groups) - len(highlighted_groups)})", GRAY, size=7)
)
for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    figure.add_trace(make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, size=10))

figure.update_layout(
    title=dict(text="Feature-name clusters: spectral clustering (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

## 5. Save the general clustering result

This cell saves which general cluster each of the 445 features ended up in, as `data/full_spectral_labels.csv`. 

The filtered notebook clusters one product type at a time. For analysing whether a filtered cluster matches a general cluster, it needs the general answer for the same features. Instead of rerunning the general clustering there, it reads this file. This also guarantees that the comparison uses the same clusters that are reported for the general run.

Rerun this cell whenever the general run changes, otherwise the comparison uses an outdated result.

In [10]:
import pandas as pd
pd.DataFrame({"Attribute": attributes, "Cluster": labels}).to_csv("data/full_spectral_labels.csv", index=False)

## 5.2 Filtered spectral clustering by product type

# 6. Evaluation

## 6.1 Setup

All methods are scored on the same distance matrix `D = 1 - S` with the helpers below. The measures are **internal**: they describe the shape of the clusters, not whether two feature names really mean the same.

- **Silhouette** (higher is better, −1 to 1): whether a feature name is closer to its own cluster than to the nearest other cluster, averaged over all feature names.
- **Medoid Davies–Bouldin-like score** (lower is better): for each cluster, the worst ratio of (scatter of two clusters) to (distance between their medoids), averaged. It is an adaptation for a precomputed distance matrix, not the standard index.
- **Modularity** (higher is better): within-cluster similarity compared with what random links would give, on the full similarity graph.
- **Noise and coverage:** to score all 445 feature names, each DBSCAN/HDBSCAN noise feature name counts as its own singleton cluster. A singleton has zero silhouette and zero scatter, which can make the Davies–Bouldin-like score look better. **Coverage** is the share of feature names in a non-noise cluster and must be read together with the scores.
- **Runtime:** one fit with the selected parameters.

| Method | Parameters tested | Selection |
|---|---|---|
| Agglomerative | average linkage, fixed `distance_threshold = 0.70` | fixed |
| DBSCAN | `eps` 0.20–0.68 by 0.02 × `min_samples` ∈ {2,3,4,5,6,8,10,12,14} (225 runs); clusters < 2 marked as noise | highest silhouette, then coverage, then fewer clusters |
| HDBSCAN | `min_cluster_size` ∈ {2,3,4,5,8,12,20} × `min_samples` ∈ {2,3,4,5,6,8}, skipping `min_samples > min_cluster_size` | same rule as DBSCAN |
| Spectral | 15 nearest neighbours; k swept 10–100 by 5 in `5_spectral.ipynb` | k = 35 (sweep and inspection) |

To compare the methods fairly, each one is re-run below with its final setting on the same distance matrix `D = 1 - S` and scored with the same helpers. This keeps the comparison independent of variables left over from sections 4 and 5.

In [1]:
import time
from collections import Counter
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import AgglomerativeClustering, DBSCAN, HDBSCAN, KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

with np.load(Path("data/similarity.npz")) as saved:
    S = saved["similarity"].astype(float)
    feature_names = saved["titles"].tolist()

n = len(feature_names)
assert S.shape == (n, n) and np.allclose(S, S.T)
D = np.clip(1 - S, 0, 1)
np.fill_diagonal(D, 0)
G = S.copy()
np.fill_diagonal(G, 0)


def score_labels(labels, distances=D):
    """Scores all feature names; DBSCAN/HDBSCAN noise (-1) becomes one singleton cluster per feature name."""
    labels = np.asarray(labels).copy()
    noise = labels == -1
    n_noise = int(noise.sum())
    full_labels = labels.copy()
    next_label = int(labels[~noise].max() + 1) if np.any(~noise) else 0
    full_labels[noise] = np.arange(next_label, next_label + n_noise)
    silhouette = float("nan")
    if 2 <= len(np.unique(full_labels)) < len(labels):
        silhouette = float(silhouette_score(distances, full_labels, metric="precomputed"))
    return {"labels": labels, "full_labels": full_labels, "clusters": len(set(labels[~noise].tolist())),
            "noise": n_noise, "coverage": float((~noise).mean()), "silhouette_all": silhouette}


def mark_small_clusters_as_noise(labels, minimum_size=2):
    labels = np.asarray(labels).copy()
    for label in np.unique(labels):
        if label != -1 and np.count_nonzero(labels == label) < minimum_size:
            labels[labels == label] = -1
    return labels


def fair_rank(row):
    # selection rule for the density grids: highest silhouette, then highest coverage, then fewer clusters
    return (-row["metrics"]["silhouette_all"], -row["metrics"]["coverage"], row["metrics"]["clusters"])


def medoid_db_like(labels, distances=D):
    """Davies-Bouldin-style score for a precomputed distance matrix, using cluster medoids (lower is better)."""
    cluster_ids = np.unique(labels)
    if len(cluster_ids) < 2:
        return float("nan")
    medoids, scatters = [], []
    for cluster_id in cluster_ids:
        members = np.flatnonzero(labels == cluster_id)
        medoid = members[np.argmin(distances[np.ix_(members, members)].sum(axis=1))]
        medoids.append(medoid)
        scatters.append(float(distances[members, medoid].mean()))
    separation = distances[np.ix_(medoids, medoids)].copy()
    np.fill_diagonal(separation, np.inf)
    scatters = np.asarray(scatters)
    ratios = (scatters[:, None] + scatters[None, :]) / np.maximum(separation, 1e-12)
    return float(ratios.max(axis=1).mean())


def modularity(labels, graph=G):
    """Weighted modularity on the full similarity graph (higher is better)."""
    degree = graph.sum(axis=1)
    two_m = degree.sum()
    same = labels[:, None] == labels[None, :]
    return float(((graph - np.outer(degree, degree) / two_m) * same).sum() / two_m)


clusterings = {}
print(f"Loaded {n} feature names; distance matrix {D.shape}")

Loaded 445 feature names; distance matrix (445, 445)


In [2]:
AGGLO_THRESHOLD = 0.70  # fixed average-linkage distance threshold, as in 4_agglomerative.ipynb

start = time.perf_counter()
agglo_labels = AgglomerativeClustering(
    n_clusters=None, metric="precomputed", linkage="average", distance_threshold=AGGLO_THRESHOLD
).fit_predict(D)
clusterings["Agglomerative"] = {"metrics": score_labels(agglo_labels), "seconds": time.perf_counter() - start,
                                "params": f"average linkage, threshold={AGGLO_THRESHOLD}"}
print(f"Agglomerative: {clusterings['Agglomerative']['metrics']['clusters']} clusters")

Agglomerative: 138 clusters


In [3]:
# Grid: eps 0.20-0.68 by 0.02, nine min_samples values; clusters smaller than 2 are relabelled as noise
DBSCAN_EPS = np.round(np.arange(0.20, 0.70, 0.02), 2).tolist()
DBSCAN_MIN_SAMPLES = [2, 3, 4, 5, 6, 8, 10, 12, 14]
dbscan_rows = []
for eps in DBSCAN_EPS:
    for min_samples in DBSCAN_MIN_SAMPLES:
        labels = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed").fit_predict(D)
        metrics = score_labels(mark_small_clusters_as_noise(labels))
        if metrics["clusters"] >= 2 and np.isfinite(metrics["silhouette_all"]):
            dbscan_rows.append({"eps": eps, "min_samples": min_samples, "metrics": metrics})
dbscan_best = sorted(dbscan_rows, key=fair_rank)[0]

# time one fit with the selected parameters
start = time.perf_counter()
mark_small_clusters_as_noise(DBSCAN(eps=dbscan_best["eps"], min_samples=dbscan_best["min_samples"], metric="precomputed").fit_predict(D))
clusterings["DBSCAN"] = {"metrics": dbscan_best["metrics"], "seconds": time.perf_counter() - start,
                         "params": f"eps={dbscan_best['eps']:.2f}, min_samples={dbscan_best['min_samples']}"}
print(f"DBSCAN grid: {len(DBSCAN_EPS) * len(DBSCAN_MIN_SAMPLES)} combinations, {len(dbscan_rows)} valid; "
      f"selected {clusterings['DBSCAN']['params']}")

dbscan_grid = np.full((len(DBSCAN_MIN_SAMPLES), len(DBSCAN_EPS)), np.nan)
for row in dbscan_rows:
    dbscan_grid[DBSCAN_MIN_SAMPLES.index(row["min_samples"]), DBSCAN_EPS.index(row["eps"])] = row["metrics"]["silhouette_all"]
figure = go.Figure(go.Heatmap(z=dbscan_grid, x=DBSCAN_EPS, y=DBSCAN_MIN_SAMPLES, colorscale="Viridis",
                              colorbar={"title": "Silhouette"},
                              hovertemplate="eps=%{x}, min_samples=%{y}<br>silhouette=%{z:.3f}<extra></extra>"))
figure.update_layout(template="simple_white", width=700, height=420, title="DBSCAN: eps × min_samples (all-feature-name silhouette)",
                     xaxis_title="eps", yaxis_title="min_samples")
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

DBSCAN grid: 225 combinations, 129 valid; selected eps=0.40, min_samples=2


Note: the selected values lie at the low edge of the tested grid.

In [4]:
# Grid: each pair is tested only when min_samples <= min_cluster_size
HDBSCAN_MIN_CLUSTER_SIZE = [2, 3, 4, 5, 8, 12, 20]
HDBSCAN_MIN_SAMPLES = [2, 3, 4, 5, 6, 8]
hdbscan_rows = []
for min_cluster_size in HDBSCAN_MIN_CLUSTER_SIZE:
    for min_samples in HDBSCAN_MIN_SAMPLES:
        if min_samples > min_cluster_size:
            continue
        labels = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, metric="precomputed", copy=True).fit_predict(D)
        metrics = score_labels(labels)
        if metrics["clusters"] >= 2 and np.isfinite(metrics["silhouette_all"]):
            hdbscan_rows.append({"min_cluster_size": min_cluster_size, "min_samples": min_samples, "metrics": metrics})
hdbscan_best = sorted(hdbscan_rows, key=fair_rank)[0]

start = time.perf_counter()
HDBSCAN(min_cluster_size=hdbscan_best["min_cluster_size"], min_samples=hdbscan_best["min_samples"],
        metric="precomputed", copy=True).fit_predict(D)
clusterings["HDBSCAN"] = {"metrics": hdbscan_best["metrics"], "seconds": time.perf_counter() - start,
                          "params": f"min_cluster_size={hdbscan_best['min_cluster_size']}, min_samples={hdbscan_best['min_samples']}"}
print(f"HDBSCAN grid: {len(hdbscan_rows)} valid pairs; selected {clusterings['HDBSCAN']['params']}")

hdbscan_grid = np.full((len(HDBSCAN_MIN_SAMPLES), len(HDBSCAN_MIN_CLUSTER_SIZE)), np.nan)
for row in hdbscan_rows:
    hdbscan_grid[HDBSCAN_MIN_SAMPLES.index(row["min_samples"]), HDBSCAN_MIN_CLUSTER_SIZE.index(row["min_cluster_size"])] = row["metrics"]["silhouette_all"]
figure = go.Figure(go.Heatmap(z=hdbscan_grid, x=HDBSCAN_MIN_CLUSTER_SIZE, y=HDBSCAN_MIN_SAMPLES, colorscale="Viridis",
                              colorbar={"title": "Silhouette"},
                              hovertemplate="min_cluster_size=%{x}, min_samples=%{y}<br>silhouette=%{z:.3f}<extra></extra>"))
figure.update_layout(template="simple_white", width=700, height=420, title="HDBSCAN: min_cluster_size × min_samples (all-feature-name silhouette)",
                     xaxis_title="min_cluster_size", yaxis_title="min_samples")
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

HDBSCAN grid: 26 valid pairs; selected min_cluster_size=2, min_samples=2


In [5]:
# Same pipeline as 5_spectral.ipynb: 15-nearest-neighbour graph, normalized Laplacian, k = 35 (chosen there in section 5.2)
K_NEIGHBORS = 15
K_CLUSTERS = 35

start = time.perf_counter()
W = G.copy()
thresholds = np.sort(W, axis=1)[:, -K_NEIGHBORS][:, None]
W = np.where((W >= thresholds) | (W >= thresholds.T), W, 0)
inverse_sqrt_degree = 1 / np.sqrt(np.maximum(W.sum(axis=1), 1e-12))
laplacian = np.eye(n) - W * np.outer(inverse_sqrt_degree, inverse_sqrt_degree)
_, eigenvectors = np.linalg.eigh(laplacian)
U = eigenvectors[:, :K_CLUSTERS]
U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
spectral_labels = KMeans(n_clusters=K_CLUSTERS, n_init=10, random_state=0).fit_predict(U)
clusterings["Spectral"] = {"metrics": score_labels(spectral_labels), "seconds": time.perf_counter() - start,
                           "params": f"k={K_CLUSTERS}, k_neighbors={K_NEIGHBORS}, seed=0"}
print(f"Spectral: {K_CLUSTERS} clusters on a graph with {int(np.count_nonzero(np.triu(W)))} edges")

Spectral: 35 clusters on a graph with 4618 edges


## 6.2 Comparison of all methods

In [6]:
rows = []
for name, result in clusterings.items():
    metrics = result["metrics"]
    rows.append({
        "method": name,
        "parameters": result["params"],
        "clusters": metrics["clusters"],
        "coverage": f"{metrics['coverage']:.1%}",
        "silhouette": round(metrics["silhouette_all"], 3),
        "medoid_db_like": round(medoid_db_like(metrics["full_labels"]), 3),
        "modularity": round(modularity(metrics["full_labels"]), 3),
        "runtime_s": round(result["seconds"], 3),
    })

import pandas as pd
comparison = pd.DataFrame(rows).set_index("method")
comparison

,parameters,clusters,coverage,silhouette,medoid_db_like,modularity,runtime_s
method,,,,,,,
Agglomerative,"average linkage, threshold=0.7",138,100.0%,0.184,0.911,0.070,0.040
DBSCAN,"eps=0.40, min_samples=2",59,37.1%,0.138,0.492,0.016,0.005
HDBSCAN,"min_cluster_size=2, min_samples=2",106,73.3%,0.178,0.725,0.030,0.018
Spectral,"k=35, k_neighbors=15, seed=0",35,100.0%,0.137,1.585,0.104,0.145


## 6.3 Qualitative examples

Each panel shows the same t-SNE projection, coloured by one method's clusters; noise feature names are grey crosses. t-SNE is for display only and can distort distances.

In [7]:
embedding = TSNE(n_components=2, metric="precomputed", init="random", learning_rate="auto",
                 perplexity=30, random_state=0).fit_transform(D)

figure = make_subplots(rows=2, cols=2, subplot_titles=[
    f"{name}: {result['metrics']['clusters']} clusters, silhouette {result['metrics']['silhouette_all']:.3f}"
    for name, result in clusterings.items()])
for index, (name, result) in enumerate(clusterings.items()):
    row, col = index // 2 + 1, index % 2 + 1
    labels = result["metrics"]["labels"]
    noise = labels == -1
    figure.add_trace(go.Scattergl(
        x=embedding[~noise, 0], y=embedding[~noise, 1], mode="markers",
        marker={"color": labels[~noise], "colorscale": "Turbo", "size": 7, "opacity": 0.8, "showscale": False},
        text=np.asarray(feature_names)[~noise], hovertemplate="%{text}<extra>cluster</extra>", showlegend=False),
        row=row, col=col)
    if noise.any():
        figure.add_trace(go.Scattergl(
            x=embedding[noise, 0], y=embedding[noise, 1], mode="markers",
            marker={"color": "#999999", "size": 6, "symbol": "x", "opacity": 0.65},
            text=np.asarray(feature_names)[noise], hovertemplate="%{text}<extra>noise</extra>", showlegend=False),
            row=row, col=col)
figure.update_xaxes(showticklabels=False)
figure.update_yaxes(showticklabels=False)
figure.update_layout(template="simple_white", height=850, width=1100, title="Feature-name clusters per method (shared t-SNE projection)")
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

## 6.4 General vs filtered spectral clustering

## 6.5 Summary

# 7. Limitations and future work